## Training functions

In [ ]:
def moving_average(arr, window_size):
    """
    Calculates the moving average of a numpy array.
    Args:
        arr: The input numpy array.
        window_size: The size of the moving average window.
      Returns:
        A numpy array containing the moving average.
    """
    if len(arr) < window_size:
        return np.array([])  # Return empty array if not enough data

    temp_sums = np.cumsum(arr)
    moving_avg = (temp_sums[window_size:] - temp_sums[:-window_size]) / window_size
    return moving_avg

def permutation_importance_ngram(model, target, tokens, numerical_features = [0], names_numerical_features = None, only_tokens = False, num_repeats = 10, verbose = 1):
    """
    Computes Permutation Importance for a neural network that takes token embeddings and numerical features.

    Parameters:
        model (nn.Module): The PyTorch model to evaluate.
        target (torch.Tensor): Ground truth labels.
        tokens (torch.Tensor): Token input (integer indices).
        numerical_features (torch.Tensor or list): Numerical features input. Default: [0]
        names_numerical_features (list): name of the numerical features. Default: None.     
        only_tokens (boolean): flag to check if the model only contains tokens or tokens + numerical features.
        num_repeats (int): Number of times to permute each feature.

    Returns:
        pd.DataFrame: DataFrame with the descritive statistics of Permutation importance for each feature.
    """
    # Ensure inputs are torch tensors
    tokens = torch.tensor(tokens, dtype = torch.long) if not isinstance(tokens, torch.Tensor) else tokens
    numerical_features = (
        torch.tensor(numerical_features, dtype=torch.float32)
        if not isinstance(numerical_features, torch.Tensor)
        else numerical_features
    )
    target = torch.tensor(target, dtype = torch.long) if not isinstance(target, torch.Tensor) else target
    # Compute baseline loss
    model.eval()
    with torch.no_grad():
        if only_tokens:
            output = model(tokens)
        else:
            output = model(tokens, numerical_features)
        baseline_loss = F.cross_entropy(output, target).item()
    if verbose:
        print(f"Baseline model loss: {baseline_loss: .4f}")
    
    # Initialize importance dictionary
    if only_tokens:
        feature_importance = {"tokens": [list()] * tokens.shape[1]}
    else:
        feature_importance = {
        "tokens": [list()] * tokens.shape[1],
        "numerical_features": [list()] * numerical_features.shape[1],
    }
        
    # Permutation importance for token features
    name_cols = ['token_' + str(i  + 1) for i in range(tokens.shape[1])][::-1]
    for col in range(tokens.shape[1]):
        losses = []
        if verbose:
            print(f"Calculating the importance of {name_cols[col]: <20}", end = "\r")
        for _ in range(num_repeats):
            permuted_tokens = tokens.clone()
            permuted_tokens[:, col] = permuted_tokens[torch.randperm(tokens.size(0)), col]
            with torch.no_grad():
                if only_tokens:
                    output = model(permuted_tokens)
                else:
                    output = model(permuted_tokens, numerical_features)
                loss = F.cross_entropy(output, target).item()
            losses.append(loss)
        feature_importance["tokens"][col] = losses
    
    # Permutation importance for numerical features
    if only_tokens == False:
        if names_numerical_features is None:
            names_numerical_features = [i for i in range(numerical_features.shape[1])]
        for col in range(numerical_features.shape[1]):
            losses = []
            if verbose:
                print(f"Calculating the importance of {names_numerical_features[col]: <20}", end = "\r")
            for _ in range(num_repeats):
                permuted_numerical = numerical_features.clone()
                permuted_numerical[:, col] = permuted_numerical[torch.randperm(numerical_features.size(0)), col]
                with torch.no_grad():
                    output = model(tokens, permuted_numerical)
                    loss = F.cross_entropy(output, target).item()
                losses.append(loss)
            feature_importance["numerical_features"][col] = losses
    feature_importance['baseline'] = baseline_loss
    
    df_importance_tokens = pd.DataFrame(feature_importance['tokens'], index = name_cols)\
    .T.describe().T
    if only_tokens == False:
        df_importance_numerical = pd.DataFrame(feature_importance['numerical_features'], 
                                               index = names_numerical_features)\
        .T.describe().T
    else:
        df_importance_numerical = pd.DataFrame()
    
    df_importance = pd.concat([df_importance_tokens, df_importance_numerical], axis = 0)\
    .assign(baseline = feature_importance['baseline'])\
    .reset_index()\
    .assign(mean = lambda x: x['mean'] - x['baseline'])\
    .assign(perc = lambda x: 100 * x['mean']/x['mean'].sum())\
    .sort_values("perc", ascending = False)\
    .assign(percacm = lambda x: x['perc'].cumsum())\
    .round(4)
    
    return df_importance

def permutation_importance_ngram(
    model,
    target,
    tokens,
    numerical_features = None,
    state_tokens = None,
    names_numerical_features = None,
    only_tokens: bool = False,
    num_repeats: int = 10,
    verbose: int = 1,
    mask_data = None
):
    """
    Computes Permutation Importance for a neural network that takes:
        - token embeddings,
        - optional numerical features,
        - optional state_token.

    Parâmetros:
        model (nn.Module): Modelo PyTorch a ser avaliado.
        target (torch.Tensor ou array-like): Rótulos verdadeiros.
        tokens (torch.Tensor ou array-like): Tokens de entrada (N, context_size).
        numerical_features (torch.Tensor ou array-like ou None): Variáveis numéricas (N, num_features).
        state_tokens (torch.Tensor ou array-like ou None): Token de estado (N,) ou (N,1).
        names_numerical_features (list ou None): nomes das variáveis numéricas.
        only_tokens (bool): se True, assume modelo que usa apenas tokens.
        num_repeats (int): número de permutações por feature.
        verbose (int): nível de verbosidade (0 = silencioso).
        mask_data (torch.Tensor, optional): Mascara booleana com a mesma dimensao dos logits, 
            onde True indica token proibido.
        

    Retorna:
        pd.DataFrame: estatísticas descritivas da importância por permutação
                        para cada feature (tokens, numéricos e state_token, se houver).
    """

    # --------------------------
    # Garantir tensores
    # --------------------------
    tokens = torch.tensor(tokens, dtype = torch.long) if not isinstance(tokens, torch.Tensor) else tokens
    target = torch.tensor(target, dtype = torch.long) if not isinstance(target, torch.Tensor) else target

    if numerical_features is not None:
        numerical_features = (
            torch.tensor(numerical_features, dtype = torch.float32)
            if not isinstance(numerical_features, torch.Tensor)
            else numerical_features
        )

    if state_tokens is not None:
        state_tokens = (
            torch.tensor(state_tokens, dtype = torch.long)
            if not isinstance(state_tokens, torch.Tensor)
            else state_tokens
        )
        # Se tiver forma (N,1), achata para (N,)
        if state_tokens.dim() == 2 and state_tokens.size(1) == 1:
            state_tokens = state_tokens.squeeze(1)

    if mask_data is not None:
        mask_data = (
            torch.tensor(mask_data, dtype = torch.bool)
            if not isinstance(mask_data, torch.Tensor)
            else mask_data
        )

    if mask_data.dtype != torch.bool:
        mask_data = mask_data.bool()

    # Flags de presença real (ignorando only_tokens)
    has_num = (numerical_features is not None) and (not only_tokens)
    has_state = (state_tokens is not None) and (not only_tokens)

    # --------------------------
    # Função auxiliar para chamar o modelo corretamente
    # --------------------------
    def forward_model(tok, num, st):
        if only_tokens:
            return model(tok)
        elif has_state and has_num:
            return model(tok, num, st)
        elif has_state and (not has_num):
            return model(tok, st)
        elif has_num and (not has_state):
            return model(tok, num)
        else:
            # fallback: só tokens
            return model(tok)
        
    def compute_masked_loss(tok, num, st, tgt, mask):
        output = forward_model(tok, num, st)

        if mask is not None:
            if mask.dtype != torch.bool:
                mask = mask.bool()

            if not torch.all(~mask[torch.arange(tgt.size(0), device = tgt.device), tgt]):
                raise ValueError("Há targets verdadeiros mascarados.")

            if not torch.all((~mask).any(dim = 1)):
                raise ValueError("Há linhas sem nenhuma classe válida na máscara.")

            output = output.masked_fill(mask, -1e9)

        loss = F.cross_entropy(output, tgt).item()
        return loss

    # --------------------------
    # Loss baseline
    # --------------------------

    model.eval()
    with torch.no_grad():
        baseline_loss = compute_masked_loss(tokens, numerical_features, state_tokens, target, mask_data)

    if verbose:
        print(f"Baseline model loss: {baseline_loss: .4f}")

    # --------------------------
    # Inicializar estruturas de importância
    # --------------------------
    feature_importance = {}
    # tokens: uma lista de listas (uma lista de losses por coluna)
    feature_importance["tokens"] = [[] for _ in range(tokens.shape[1])]

    if has_num:
        feature_importance["numerical_features"] = [[] for _ in range(numerical_features.shape[1])]
    else:
        feature_importance["numerical_features"] = []

    if has_state:
        # único “feature” para o state_token
        feature_importance["state"] = []  # lista de losses

    # --------------------------
    # Importância por permutação - TOKENS
    # --------------------------
    name_cols = ['token_' + str(i + 1) for i in range(tokens.shape[1])][::-1]

    for col in range(tokens.shape[1]):
        losses = []
        if verbose:
            print(f"Calculating the importance of {name_cols[col]: <20}", end="\r")

        for _ in range(num_repeats):
            permuted_tokens = tokens.clone()
            permuted_tokens[:, col] = permuted_tokens[torch.randperm(tokens.size(0)), col]

            with torch.no_grad():
                loss = compute_masked_loss(permuted_tokens, numerical_features, state_tokens, target, mask_data)

            losses.append(loss)

        feature_importance["tokens"][col] = losses

    # --------------------------
    # Importância por permutação - NUMÉRICOS
    # --------------------------
    if has_num:
        if names_numerical_features is None:
            names_numerical_features = [f"num_{i}" for i in range(numerical_features.shape[1])]

        for col in range(numerical_features.shape[1]):
            losses = []
            if verbose:
                print(f"Calculating the importance of {names_numerical_features[col]: <20}", end="\r")

            for _ in range(num_repeats):
                permuted_numerical = numerical_features.clone()
                permuted_numerical[:, col] = permuted_numerical[torch.randperm(numerical_features.size(0)), col]

                with torch.no_grad():
                    loss = compute_masked_loss(tokens, permuted_numerical, state_tokens, target, mask_data)

                losses.append(loss)

            feature_importance["numerical_features"][col] = losses

    # --------------------------
    # Importância por permutação - STATE TOKEN
    # --------------------------
    if has_state:
        losses = []
        if verbose:
            print(f"Calculating the importance of state_token     ", end="\r")

        for _ in range(num_repeats):
            perm = torch.randperm(state_tokens.size(0))
            permuted_state = state_tokens[perm]

            with torch.no_grad():
                loss = compute_masked_loss(tokens, numerical_features, permuted_state, target, mask_data)

            losses.append(loss)

        feature_importance["state"] = losses

    feature_importance["baseline"] = baseline_loss

    # --------------------------
    # Construir DataFrames
    # --------------------------
    # Tokens
    df_importance_tokens = (
        pd.DataFrame(feature_importance["tokens"], index=name_cols)
        .T.describe()
        .T
    )

    # Numéricos
    if has_num:
        df_importance_numerical = (
            pd.DataFrame(feature_importance["numerical_features"], index=names_numerical_features)
            .T.describe()
            .T
        )
    else:
        df_importance_numerical = pd.DataFrame()

    # State token
    if has_state:
        df_importance_state = (
            pd.DataFrame([feature_importance["state"]], index=["state_token"])
            .T.describe()
            .T
        )
    else:
        df_importance_state = pd.DataFrame()

    # --------------------------
    # Concatenar tudo
    # --------------------------
    dfs_to_concat = [df_importance_tokens]
    if not df_importance_state.empty:
        dfs_to_concat.append(df_importance_state)
    if not df_importance_numerical.empty:
        dfs_to_concat.append(df_importance_numerical)

    df_importance = (
        pd.concat(dfs_to_concat, axis=0)
        .assign(baseline = feature_importance["baseline"])
        .reset_index()
        .rename(columns={"index": "feature"})
        .assign(mean = lambda x: x["mean"] - x["baseline"])
    )

    # Se todas as médias derem zero (caso extremo), evitar divisão por zero
    total_mean = df_importance["mean"].sum()
    if total_mean == 0:
        df_importance["perc"] = 0.0
    else:
        df_importance["perc"] = 100 * df_importance["mean"] / total_mean

    df_importance = (
        df_importance
        .sort_values("perc", ascending=False)
        .assign(percacm = lambda x: x["perc"].cumsum())
        .round(4)
    )

    return df_importance

def permutation_importance_time_tower(
    model,
    target,
    tokens,
    time_diff_features = None,
    numerical_features = None,
    names_time_features = None,
    names_numerical_features = None,
    only_tokens: bool = False,
    num_repeats: int = 10,
    verbose: int = 1,
    mask_data = None
):
    """
    Computes Permutation Importance para redes no estilo *TimeDiffTower* que recebem:
      - tokens (obrigatório)
      - time_diff_features (opcional, ex: remaining_time, score_diff)
      - numerical_features (opcional, demais variáveis numéricas)

    A função tenta ser o mais parecida possível com a permutation_importance_ngram original.

    Parâmetros:
        model (nn.Module): Modelo PyTorch a ser avaliado.
        target: rótulos verdadeiros (Tensor ou array-like).
        tokens: Tensor ou array-like, shape (N, context_size).
        time_diff_features: Tensor ou array-like ou None, shape (N, T_time).
        numerical_features: Tensor ou array-like ou None, shape (N, T_num).
        names_time_features: lista de nomes para as colunas de tempo/diff.
        names_numerical_features: lista de nomes para as colunas numéricas.
        only_tokens (bool): se True, ignora time_diff e numéricos na chamada do modelo.
        num_repeats (int): nº de permutações por feature.
        verbose (int): nível de verbosidade.
        mask_data (torch.Tensor, optional): Mascara booleana com a mesma dimensao dos logits, 
            onde True indica token proibido.

    Retorna:
        pd.DataFrame com estatísticas de importância (tokens, tempo, numéricos).
    """

    # --------------------------
    # Garantir tensores
    # --------------------------
    tokens = torch.tensor(tokens, dtype = torch.long) if not isinstance(tokens, torch.Tensor) else tokens
    target = torch.tensor(target, dtype = torch.long) if not isinstance(target, torch.Tensor) else target

    if time_diff_features is not None:
        time_diff_features = (
            torch.tensor(time_diff_features, dtype = torch.float32)
            if not isinstance(time_diff_features, torch.Tensor)
            else time_diff_features
        )

    if numerical_features is not None:
        numerical_features = (
            torch.tensor(numerical_features, dtype  =torch.float32)
            if not isinstance(numerical_features, torch.Tensor)
            else numerical_features
        )

    if mask_data is not None:
        mask_data = (
            torch.tensor(mask_data, dtype = torch.bool)
            if not isinstance(mask_data, torch.Tensor)
            else mask_data
        )
        if mask_data.dtype != torch.bool:
            mask_data = mask_data.bool()

    # Flags de presença real (ignorando only_tokens)
    has_time = (time_diff_features is not None) and (not only_tokens)
    has_num  = (numerical_features is not None) and (not only_tokens)

    # --------------------------
    # Garantir que tudo está no mesmo device do modelo
    # --------------------------
    model_device = next(model.parameters()).device

    tokens = tokens.to(model_device)
    target = target.to(model_device)
    if has_time:
        time_diff_features = time_diff_features.to(model_device)
    if has_num:
        numerical_features = numerical_features.to(model_device)

    if mask_data is not None:
        mask_data = mask_data.to(model_device)

    # --------------------------
    # Função auxiliar para chamar o modelo
    # --------------------------
    def forward_model(tok, time_f, num_f):
        if only_tokens:
            return model(tok)
        elif has_time and has_num:
            return model(tok, time_f, num_f)
        elif has_time and (not has_num):
            return model(tok, time_f, None if "numerical_features" in model.__class__.__dict__ else time_f)
        elif has_num and (not has_time):
            # modelo espera (tokens, time_diff, num) → podemos passar zeros em time_diff
            # ou assumir que ele aceita None; aqui vamos passar None e assumir que o forward lida
            return model(tok, None, num_f)
        else:
            # fallback: só tokens
            return model(tok)

    def compute_masked_loss(tok, time_f, num_f, tgt, mask):
        output = forward_model(tok, time_f, num_f)

        if mask is not None:
            if not torch.all(~mask[torch.arange(tgt.size(0), device=tgt.device), tgt]):
                raise ValueError("Há targets verdadeiros mascarados.")

            if not torch.all((~mask).any(dim=1)):
                raise ValueError("Há linhas sem nenhuma classe válida na máscara.")

            output = output.masked_fill(mask, -1e9)

        loss = F.cross_entropy(output, tgt).item()
        return loss
    
    # --------------------------
    # Loss baseline
    # --------------------------
    model.eval()
    with torch.no_grad():
        baseline_loss = compute_masked_loss(
            tokens, time_diff_features, numerical_features, target, mask_data
        )

    if verbose:
        print(f"Baseline model loss: {baseline_loss: .4f}")

    # --------------------------
    # Inicializar estruturas de importância
    # --------------------------
    feature_importance = {}

    # Tokens: lista de listas
    feature_importance["tokens"] = [[] for _ in range(tokens.shape[1])]

    # Time/diff
    if has_time:
        feature_importance["time_features"] = [[] for _ in range(time_diff_features.shape[1])]
    else:
        feature_importance["time_features"] = []

    # Numéricos
    if has_num:
        feature_importance["numerical_features"] = [[] for _ in range(numerical_features.shape[1])]
    else:
        feature_importance["numerical_features"] = []

    # --------------------------
    # Importância - TOKENS
    # --------------------------
    name_cols = ['token_' + str(i + 1) for i in range(tokens.shape[1])][::-1]

    for col in range(tokens.shape[1]):
        losses = []
        if verbose:
            print(f"Calculating the importance of {name_cols[col]: <20}", end="\r")

        for _ in range(num_repeats):
            permuted_tokens = tokens.clone()
            permuted_tokens[:, col] = permuted_tokens[torch.randperm(tokens.size(0)), col]

            with torch.no_grad():
                loss = compute_masked_loss(
                    permuted_tokens, time_diff_features, numerical_features, target, mask_data
                )

            losses.append(loss)

        feature_importance["tokens"][col] = losses

    # --------------------------
    # Importância - TIME/DIFF
    # --------------------------
    if has_time:
        if names_time_features is None:
            names_time_features = [f"time_{i}" for i in range(time_diff_features.shape[1])]

        for col in range(time_diff_features.shape[1]):
            losses = []
            if verbose:
                print(f"Calculating the importance of {names_time_features[col]: <20}", end="\r")

            for _ in range(num_repeats):
                perm_time = time_diff_features.clone()
                perm_time[:, col] = perm_time[torch.randperm(time_diff_features.size(0)), col]

                with torch.no_grad():
                    loss = compute_masked_loss(
                        tokens, perm_time, numerical_features, target, mask_data
                    )

                losses.append(loss)

            feature_importance["time_features"][col] = losses

    # --------------------------
    # Importância - NUMÉRICOS
    # --------------------------
    if has_num:
        if names_numerical_features is None:
            names_numerical_features = [f"num_{i}" for i in range(numerical_features.shape[1])]

        for col in range(numerical_features.shape[1]):
            losses = []
            if verbose:
                print(f"Calculating the importance of {names_numerical_features[col]: <20}", end="\r")

            for _ in range(num_repeats):
                perm_num = numerical_features.clone()
                perm_num[:, col] = perm_num[torch.randperm(numerical_features.size(0)), col]

                with torch.no_grad():
                    loss = compute_masked_loss(
                        tokens, time_diff_features, perm_num, target, mask_data
                    )

                losses.append(loss)

            feature_importance["numerical_features"][col] = losses

    feature_importance["baseline"] = baseline_loss

    # --------------------------
    # Construir DataFrames por grupo
    # --------------------------
    # Tokens
    df_importance_tokens = (
        pd.DataFrame(feature_importance["tokens"], index=name_cols)
        .T.describe()
        .T
    )

    # Time/diff
    if has_time:
        df_importance_time = (
            pd.DataFrame(feature_importance["time_features"], index=names_time_features)
            .T.describe()
            .T
        )
    else:
        df_importance_time = pd.DataFrame()

    # Numéricos
    if has_num:
        df_importance_numerical = (
            pd.DataFrame(feature_importance["numerical_features"], index=names_numerical_features)
            .T.describe()
            .T
        )
    else:
        df_importance_numerical = pd.DataFrame()

    # --------------------------
    # Concatenar tudo
    # --------------------------
    dfs_to_concat = [df_importance_tokens]
    if not df_importance_time.empty:
        dfs_to_concat.append(df_importance_time)
    if not df_importance_numerical.empty:
        dfs_to_concat.append(df_importance_numerical)

    df_importance = (
        pd.concat(dfs_to_concat, axis=0)
        .assign(baseline = feature_importance["baseline"])
        .reset_index()
        .rename(columns={"index": "feature"})
        .assign(mean = lambda x: x["mean"] - x["baseline"])
    )

    total_mean = df_importance["mean"].sum()
    if total_mean == 0:
        df_importance["perc"] = 0.0
    else:
        df_importance["perc"] = 100 * df_importance["mean"] / total_mean

    df_importance = (
        df_importance
        .sort_values("perc", ascending = False)
        .assign(percacm = lambda x: x["perc"].cumsum())
        .round(4)
    )

    return df_importance

def permutation_importance_ngram_multi_towers(
    model,
    target,
    tokens,
    numerical_groups: Optional[List[torch.Tensor]] = None,
    names_groups: Optional[List[List[str]]] = None,
    only_tokens: bool = False,
    num_repeats: int = 10,
    verbose: int = 1,
    mask_data = None
):
    """
    Permutation Importance para modelos no estilo MultiTowers que recebem:
      - tokens (obrigatório)
      - numerical_groups (lista de tensores), opcional se only_tokens=True

    Parâmetros:
        model: nn.Module
        target: (N,) labels (Tensor ou array-like)
        tokens: (N, context_size) (Tensor ou array-like)
        numerical_groups: lista [G1, ..., GK], cada Gi (N, d_i)
        names_groups: lista de listas com nomes de features por grupo.
                     Ex: [["rem_time","score_diff"], ["flag1","flag2",...]]
        only_tokens: se True, ignora numerical_groups e chama model(tokens)
        num_repeats: nº de permutações por feature
        verbose: 0/1
        mask_data (torch.Tensor, optional): Mascara booleana com a mesma dimensao dos logits, 
            onde True indica token proibido.


    Retorna:
        pd.DataFrame com estatísticas por feature.
    """

    # --------------------------
    # Garantir tensores
    # --------------------------
    tokens = torch.tensor(tokens, dtype=torch.long) if not isinstance(tokens, torch.Tensor) else tokens
    target = torch.tensor(target, dtype=torch.long) if not isinstance(target, torch.Tensor) else target

    has_groups = (numerical_groups is not None) and (not only_tokens)

    if has_groups:
        if not isinstance(numerical_groups, (list, tuple)):
            raise TypeError("numerical_groups deve ser lista/tupla de tensores.")
        if len(numerical_groups) == 0:
            raise ValueError("numerical_groups não pode ser lista vazia quando only_tokens=False.")

        numerical_groups = [
            torch.tensor(g, dtype=torch.float32) if not isinstance(g, torch.Tensor) else g
            for g in numerical_groups
        ]

    if mask_data is not None:
        mask_data = (
            torch.tensor(mask_data, dtype=torch.bool)
            if not isinstance(mask_data, torch.Tensor) else mask_data
        )
        if mask_data.dtype != torch.bool:
            mask_data = mask_data.bool()

    # --------------------------
    # Checar shapes
    # --------------------------
    N = tokens.size(0)
    if target.size(0) != N:
        raise ValueError("target deve ter o mesmo N de tokens.")

    if has_groups:
        for gi, g in enumerate(numerical_groups):
            if g.size(0) != N:
                raise ValueError(f"O grupo {gi} tem N diferente de tokens.")

    # --------------------------
    # Device do modelo
    # --------------------------
    model_device = next(model.parameters()).device
    tokens = tokens.to(model_device)
    target = target.to(model_device)
    if has_groups:
        numerical_groups = [g.to(model_device) for g in numerical_groups]

    if mask_data is not None:
        mask_data = mask_data.to(model_device)

    # --------------------------
    # Forward helper
    # --------------------------
    def forward_model(tok, groups):
        if only_tokens:
            return model(tok)
        return model(tok, groups)

    def compute_masked_loss(tok, groups, tgt, mask):
        out = forward_model(tok, groups)

        if mask is not None:
            if not torch.all(~mask[torch.arange(tgt.size(0), device=tgt.device), tgt]):
                raise ValueError("Há targets verdadeiros mascarados.")

            if not torch.all((~mask).any(dim=1)):
                raise ValueError("Há linhas sem nenhuma classe válida na máscara.")

            out = out.masked_fill(mask, -1e9)

        loss = F.cross_entropy(out, tgt).item()
        return loss    

    # --------------------------
    # Baseline loss
    # --------------------------
    model.eval()
    with torch.no_grad():
        baseline_loss = compute_masked_loss(
            tokens,
            numerical_groups if has_groups else None,
            target,
            mask_data
        )

    if verbose:
        print(f"Baseline model loss: {baseline_loss: .4f}")

    # --------------------------
    # Estruturas de importância
    # --------------------------
    feature_importance = {}
    feature_importance["tokens"] = [[] for _ in range(tokens.shape[1])]

    # para grupos numéricos: lista de grupos; cada grupo = lista de colunas; cada coluna = lista losses
    if has_groups:
        group_importance = []
        for g in numerical_groups:
            group_importance.append([[] for _ in range(g.shape[1])])
        feature_importance["numerical_groups"] = group_importance
    else:
        feature_importance["numerical_groups"] = []

    # --------------------------
    # TOKENS
    # --------------------------
    token_names = [f"token_{i+1}" for i in range(tokens.shape[1])][::-1]

    for col in range(tokens.shape[1]):
        losses = []
        if verbose:
            print(f"Calculating the importance of {token_names[col]: <25}", end="\r")

        for _ in range(num_repeats):
            permuted_tokens = tokens.clone()
            permuted_tokens[:, col] = permuted_tokens[torch.randperm(N, device=tokens.device), col]

            with torch.no_grad():
                loss = compute_masked_loss(
                    permuted_tokens,
                    numerical_groups if has_groups else None,
                    target,
                    mask_data
                )                

            losses.append(loss)

        feature_importance["tokens"][col] = losses

    # --------------------------
    # GRUPOS NUMÉRICOS
    # --------------------------
    if has_groups:
        # nomes default se não vierem
        if names_groups is None:
            names_groups = []
            for gi, g in enumerate(numerical_groups):
                names_groups.append([f"g{gi}_feat{j}" for j in range(g.shape[1])])
        else:
            if len(names_groups) != len(numerical_groups):
                raise ValueError("names_groups deve ter o mesmo nº de grupos de numerical_groups.")
            for gi, (g, names_g) in enumerate(zip(numerical_groups, names_groups)):
                if len(names_g) != g.shape[1]:
                    raise ValueError(f"names_groups[{gi}] deve ter tamanho {g.shape[1]}.")

        for gi, g in enumerate(numerical_groups):
            for col in range(g.shape[1]):
                losses = []
                feat_name = names_groups[gi][col]
                if verbose:
                    print(f"Calculating the importance of {feat_name: <25}", end="\r")

                for _ in range(num_repeats):
                    perm_groups = [x.clone() for x in numerical_groups]
                    perm = torch.randperm(N, device=g.device)
                    perm_groups[gi][:, col] = perm_groups[gi][perm, col]

                    with torch.no_grad():
                        loss = compute_masked_loss(
                            tokens,
                            perm_groups,
                            target,
                            mask_data
                        )                        

                    losses.append(loss)

                feature_importance["numerical_groups"][gi][col] = losses

    feature_importance["baseline"] = baseline_loss

    # --------------------------
    # DataFrames
    # --------------------------
    df_tokens = pd.DataFrame(feature_importance["tokens"], index=token_names).T.describe().T
    df_tokens = df_tokens.reset_index().rename(columns={"index": "feature"})

    dfs = [df_tokens]

    if has_groups:
        # achatar todos os grupos num df único
        rows = []
        row_names = []
        for gi in range(len(numerical_groups)):
            for col in range(numerical_groups[gi].shape[1]):
                rows.append(feature_importance["numerical_groups"][gi][col])
                row_names.append(names_groups[gi][col])

        df_num = pd.DataFrame(rows, index=row_names).T.describe().T
        df_num = df_num.reset_index().rename(columns={"index": "feature"})
        dfs.append(df_num)

    df_importance = pd.concat(dfs, axis=0, ignore_index=True)
    df_importance["baseline"] = baseline_loss
    df_importance["mean"] = df_importance["mean"] - df_importance["baseline"]

    total_mean = df_importance["mean"].sum()
    if total_mean == 0:
        df_importance["perc"] = 0.0
    else:
        df_importance["perc"] = 100 * df_importance["mean"] / total_mean

    df_importance = (
        df_importance
        .sort_values("perc", ascending=False)
        .assign(percacm=lambda x: x["perc"].cumsum())
        .round(4)
    )

    return df_importance

class NGramModel(nn.Module):
    """
    A Neural Network for N-Gram Language Modeling using a customizable 
    number of layers, activation functions, Layer Normalization, and Dropout.

    Attributes:
        embedding (torch.nn.Embedding): Embeds token indices to dense vectors.
        flatten (torch.nn.Flatten): Flattens the output of the embedding layer.
        network (torch.nn.Sequential): Sequential container of layers.

    Parameters:
        vocab_size (int): The size of the vocabulary.
        context_size (int): The number of tokens in the context.
        embedding_dim (int): The dimensionality of the embedding space.
        num_hidden_neurons (int or list): The number of neurons in each hidden layer.
                                          If a list, specifies neurons for each layer.
        num_layers (int): The number of hidden layers in the network.
        activation (callable): The activation function used in the hidden layers.
        dropout_rate (float): Dropout probability (0.0 means no dropout).
        use_layer_norm (bool): Whether to apply Layer Normalization in the layers.
    """
    def __init__(self, vocab_size, context_size, embedding_dim, num_hidden_neurons, 
                 num_layers, activation = nn.Tanh, dropout_rate = 0.0, use_layer_norm = False):
        super(NGramModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        # Building the network dynamically based on the number of layers
        layers = self._init_layers(context_size, vocab_size, embedding_dim, num_hidden_neurons, 
                                   num_layers, activation, dropout_rate, use_layer_norm)

        # Combine layers into a Sequential module
        self.network = nn.Sequential(*layers)

    def _init_layers(self, context_size, vocab_size, embedding_dim, num_hidden_neurons, 
                     num_layers, activation, dropout_rate, use_layer_norm):
        """
        Initializes the layers of the network based on the provided specifications.
        """
        layers = []
        input_size = context_size * embedding_dim

        # Handle num_hidden_neurons as a list or single value
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Append the first layer with optional LayerNorm and Dropout
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))        
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Append subsequent hidden layers with optional LayerNorm and Dropout
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))            
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Add final output layer
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))
        return layers

    def forward(self, x):
        """Defines the forward pass of the model."""
        x = self.embedding(x)
        x = self.flatten(x)
        return self.network(x)

def train_ngram_model(
    model: torch.nn.Module,
    train_data: torch.Tensor,
    train_targets: torch.Tensor,
    batch_size: int,
    n_epochs: int,
    lr: float = 0.1,
    val_data: Optional[torch.Tensor] = None,
    val_targets: Optional[torch.Tensor] = None,
    params_optimizer: Optional[dict] = None,
    device: Optional[torch.device] = None,
) -> Tuple[torch.nn.Module, List[List[float]], List[float]]:
    """
    Treina um NGramModel com esquema de épocas + batches.

    Args:
        model: modelo a treinar.
        train_data: tensor de entradas de treino (N, ...).
        train_targets: rótulos de treino (N,).
        batch_size: tamanho do batch.
        n_epochs: número de épocas.
        lr: learning rate inicial.
        val_data: dados de validação (opcional).
        val_targets: rótulos de validação (opcional).
        params_optimizer: dict com 'factor', 'threshold', 'patience' para o scheduler.
        device: torch.device("cuda") ou "cpu" (opcional).

    Returns:
        model treinado,
        train_losses: lista de épocas, cada época contém lista de losses por batch,
        val_losses: lista de losses de validação por época (se houver validação).
    """
    if params_optimizer is None:
        params_optimizer = {'factor': 0.1, 'threshold': 1e-4, 'patience': 1}

    # --------------------------
    # Device
    # --------------------------
    if device is not None:
        model = model.to(device)
        train_data = train_data.to(device)
        train_targets = train_targets.to(device)
        if val_data is not None and val_targets is not None:
            val_data = val_data.to(device)
            val_targets = val_targets.to(device)

    # --------------------------
    # Otimizador e scheduler
    # --------------------------
    optimizer = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = 1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = params_optimizer['factor'],
        threshold = params_optimizer['threshold'],
        patience = params_optimizer['patience']
    )

    num_samples = train_data.size(0)
    train_losses: List[List[float]] = []  # cada item = lista de losses por batch na época
    val_losses: List[float] = []

    best_val_loss = float("inf")
    best_state_dict = None

    # --------------------------
    # Loop de épocas
    # --------------------------
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_losses: List[float] = []

        # Embaralha índices a cada época
        perm = torch.randperm(num_samples, device = train_data.device)

        # Loop de batches
        for start in range(0, num_samples, batch_size):
            end = start + batch_size
            batch_idx = perm[start : end]

            x_batch = train_data[batch_idx]
            y_batch = train_targets[batch_idx]

            optimizer.zero_grad()
            logits = model(x_batch)
            loss = F.cross_entropy(logits, y_batch)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        train_losses.append(epoch_losses)
        epoch_train_loss = sum(epoch_losses) / len(epoch_losses)

        # --------------------------
        # Validação 1x por época
        # --------------------------
        if val_data is not None and val_targets is not None:
            model.eval()
            with torch.no_grad():
                val_logits = model(val_data)
                val_loss = F.cross_entropy(val_logits, val_targets).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)

            # guarda melhor modelo
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

            current_lr = optimizer.param_groups[0]['lr']

            # early stopping simples baseado em LR mínimo
            if current_lr <= 1e-7:
                if best_state_dict is not None:
                    model.load_state_dict(best_state_dict)
                break

            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f} | "
                f"LR: {current_lr:.2e}",
                end = "\r",
                flush = True,
            )
        else:
            # Se não tiver validação, pode logar só treino se quiser
            print(f"[Epoch {epoch:03d}] Train Loss: {epoch_train_loss:.5f}", end="\r", flush=True)
            pass

    # Se teve validação, volta pro melhor estado
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

def predict_ngram_model(model, data, k = 0):
    """
    Generates predictions from an NGramModel.

    Parameters:
        model (NGramModel): The model to generate predictions from.
        data (torch.Tensor): Input data tensor.
        k (int): The number of top probabilities to return.

    Returns:
        dict or int: Top k predictions and their probabilities if top_k 
        is specified; otherwise, the most probable class.
    """
    model.eval()
    logits = model(data)
    probabilities = torch.softmax(logits, dim = 1)
    if k:
        topk = torch.topk(probabilities, k, dim = 1)
        topk_preds = {'indices': list(topk.indices.detach().numpy()[0]),
                      'probs': list(topk.values.detach().numpy()[0])}
        return topk_preds
    else:
        if data.shape[0] == 1:
            return torch.multinomial(probabilities, 1).item()
        else:
            return [torch.multinomial(i, 1).item() for i in probabilities]
        
class NGramModel_ExtraInfo(nn.Module):
    """
    A Neural Network designed for N-Gram modeling that incorporates additional 
    numerical input features into the model architecture alongside the standard 
    token embeddings, with optional Layer Normalization and Dropout.

    Attributes:
        embedding (nn.Embedding): Embedding layer to convert token indices into 
        dense vectors.
        flatten (nn.Flatten): Flattens the output of the embedding layer to 
        prepare for linear layers.
        network (nn.Sequential): A sequence of linear and activation layers 
        culminating in the output layer.

    Parameters:
        context_size (int): The number of tokens used for context in embeddings.
        vocab_size (int): The size of the vocabulary in the embedding layer.
        embedding_dim (int): The dimensionality of the embedding vectors.
        numerical_input_dim (int): The dimensionality of the numerical feature vectors.
        num_hidden_neurons (int or list): The number of neurons in each hidden layer.
                                          If a list, specifies neurons for each layer.
        num_layers (int): The total number of hidden layers in the model.
        activation (callable): The activation function used in the network.
        dropout_rate (float): Dropout probability (0.0 means no dropout).
        use_layer_norm (bool): Whether to apply Layer Normalization in the layers.
    """
    def __init__(self, context_size, vocab_size, embedding_dim, numerical_input_dim, 
                 num_hidden_neurons, num_layers, activation = nn.Tanh, dropout_rate = 0.0, use_layer_norm = False):
        super(NGramModel_ExtraInfo, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        # Initialize the network layers
        layers = self._init_layers(context_size, vocab_size, embedding_dim, numerical_input_dim, 
                                   num_hidden_neurons, num_layers, activation, dropout_rate, use_layer_norm)

        # Use nn.Sequential to manage the network layers
        self.network = nn.Sequential(*layers)

    def _init_layers(self, context_size, vocab_size, embedding_dim, numerical_input_dim, 
                     num_hidden_neurons, num_layers, activation, dropout_rate, use_layer_norm):
        """
        Initializes the layers of the network based on the provided specifications.
        """
        layers = []
        input_size = embedding_dim * context_size + numerical_input_dim

        # Handle num_hidden_neurons as a list or single value
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Append the first layer with optional LayerNorm and Dropout
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Append subsequent hidden layers with optional LayerNorm and Dropout
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]
            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))            
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Output layer
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))
        return layers

    def forward(self, train_token, train_numerical):
        """
        Forward pass of the model.

        Parameters:
            train_token (torch.Tensor): Tensor of token indices.
            train_numerical (torch.Tensor): Tensor of numerical features.

        Returns:
            torch.Tensor: The output logits from the final layer of the network.
        """
        # Embed and flatten token inputs
        embedded = self.embedding(train_token)
        flattened = self.flatten(embedded)

        # Concatenate embedded tokens with numerical inputs
        concatenated = torch.cat((flattened, train_numerical), dim=1)

        # Pass through the network and return output
        return self.network(concatenated)

def train_num_ngram_model(
    model: torch.nn.Module,
    train_token_data: torch.Tensor,
    train_numerical_data: torch.Tensor,
    train_targets: torch.Tensor,
    batch_size: int,
    n_epochs: int,
    train_mask_data: Optional[torch.Tensor] = None,
    lr: float = 0.1,
    val_token_data: Optional[torch.Tensor] = None,
    val_numerical_data: Optional[torch.Tensor] = None,
    val_targets: Optional[torch.Tensor] = None,
    val_mask_data: Optional[torch.Tensor] = None,
    params_optimizer: Optional[dict] = None,
    device: Optional[torch.device] = None,
) -> Tuple[torch.nn.Module, List[float], List[float]]:
    """
    Treina um modelo N-gram com parte token e parte numérica.

    Args:
        model: modelo PyTorch.
        train_token_data: tensor de índices de tokens de treino (N, ...).
        train_numerical_data: tensor de variáveis numéricas de treino (N, ...).
        train_targets: rótulos de treino (N,).
        train_mask_data: máscara de treino (N, vocab_size), onde True indica token proibido (opcional).
        batch_size: tamanho do batch.
        n_epochs: número de épocas.
        lr: learning rate inicial.
        val_token_data: tokens de validação (opcional).
        val_numerical_data: numéricos de validação (opcional).
        val_targets: rótulos de validação (opcional).
        val_mask_data: máscara de validação (N, vocab_size), onde True indica token proibido (opcional).
        params_optimizer: dict com 'factor', 'threshold', 'patience'.
        device: torch.device("cuda") ou "cpu", se quiser mover tudo.
    Returns:
        model treinado, lista de train_losses (por batch), lista de val_losses (por época).
    """
    if params_optimizer is None:
        params_optimizer = {'factor': 0.1, 'threshold': 1e-4, 'patience': 1}

    # --------------------------
    # Device
    # --------------------------
    if device is not None:
        model = model.to(device)
        train_token_data = train_token_data.to(device)
        train_numerical_data = train_numerical_data.to(device)
        train_targets = train_targets.to(device)

        if train_mask_data is not None:
            train_mask_data = train_mask_data.to(device)

        if (
            val_token_data is not None
            and val_numerical_data is not None
            and val_targets is not None
        ):
            val_token_data = val_token_data.to(device)
            val_numerical_data = val_numerical_data.to(device)
            val_targets = val_targets.to(device)

            if val_mask_data is not None:
                val_mask_data = val_mask_data.to(device)            

    # Garantir dtype booleano das máscaras
    if train_mask_data is not None and train_mask_data.dtype != torch.bool:
        train_mask_data = train_mask_data.bool()

    if val_mask_data is not None and val_mask_data.dtype != torch.bool:
        val_mask_data = val_mask_data.bool()

    # --------------------------
    # Otimizador e scheduler
    # --------------------------
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr = lr,
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = params_optimizer['factor'],
        threshold = params_optimizer['threshold'],
        patience = params_optimizer['patience']
    )

    num_samples = train_token_data.size(0)
    train_losses: List[float] = []
    val_losses: List[float] = []

    best_val_loss = float("inf")
    best_state_dict = None

    # --------------------------
    # Loop de épocas
    # --------------------------
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_losses = []

        # Embaralhar índices
        perm = torch.randperm(num_samples, device = train_token_data.device)

        for idx_start in range(0, num_samples, batch_size):
            idx_end = idx_start + batch_size
            batch_idx = perm[idx_start:idx_end]

            x_tok = train_token_data[batch_idx]
            x_num = train_numerical_data[batch_idx]
            y = train_targets[batch_idx]

            optimizer.zero_grad()
            logits = model(x_tok, x_num)

            if train_mask_data is not None:
                mask_batch = train_mask_data[batch_idx]

                # Segurança: o target verdadeiro não pode estar mascarado
                if not torch.all(~mask_batch[torch.arange(y.size(0), device = y.device), y]):
                    raise ValueError("Há targets verdadeiros mascarados no batch de treino.")

                # Segurança: cada linha deve ter pelo menos uma classe válida
                if not torch.all((~mask_batch).any(dim = 1)):
                    raise ValueError("Há linhas sem nenhuma classe válida no batch de treino.")

                logits = logits.masked_fill(mask_batch, -1e9)

            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            loss_value = loss.item()
            epoch_losses.append(loss_value)
            
        train_losses.append(epoch_losses)
        epoch_train_loss = sum(epoch_losses) / len(epoch_losses)

        # --------------------------
        # Validação
        # --------------------------
        if (
            val_token_data is not None
            and val_numerical_data is not None
            and val_targets is not None
        ):
            model.eval()
            with torch.no_grad():
                val_logits = model(val_token_data, val_numerical_data)
                if val_mask_data is not None:
                    # Segurança: o target verdadeiro não pode estar mascarado
                    if not torch.all(~val_mask_data[torch.arange(val_targets.size(0), device=val_targets.device), val_targets]):
                        raise ValueError("Há targets verdadeiros mascarados nos dados de validação.")

                    # Segurança: cada linha deve ter pelo menos uma classe válida
                    if not torch.all((~val_mask_data).any(dim=1)):
                        raise ValueError("Há linhas sem nenhuma classe válida nos dados de validação.")

                    val_logits = val_logits.masked_fill(val_mask_data, -1e9)

                val_loss = F.cross_entropy(val_logits, val_targets).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

            current_lr = optimizer.param_groups[0]['lr']

            # Early stopping baseado no LR mínimo
            if current_lr <= 1e-7:
                if best_state_dict is not None:
                    model.load_state_dict(best_state_dict)
                break

            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f} | "
                f"LR: {current_lr:.2e}",
                end = "\r",
                flush = True,
            )
        else:
            # Se não tiver validação, pode logar só treino se quiser
            print(
                f"[Epoch {epoch:03d}] Train Loss: {epoch_train_loss:.5f}",
                end="\r",
                flush=True,
            )

    # Se teve validação, garantir que o melhor modelo seja carregado
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

def predict_num_ngram_model(model, tokens_data, numerical_data, k = 0, extra_info = True, mask_data = None):
    """
    Generates predictions from an NGramModel_ExtraInfo.

    Parameters:
        model (NGramModel_ExtraInfo): The model to generate predictions from.
        tokens_data (torch.Tensor): Input token data tensor.
        numerical_data (torch.Tensor): Input numerical data tensor.
        k (int): The number of top probabilities to return.
        extra_info (bool): Whether to return extra information of probabilities or not.
        mask_data (torch.Tensor, optional): Boolean mask with same shape as logits,
            where True indicates forbidden classes.

    Returns:
        dict or int: Top k predictions and their probabilities if top_k is specified; 
        otherwise, the most probable class.
    """

    model.eval()
    with torch.no_grad():
        logits = model(tokens_data, numerical_data)
        logits[:, 0] = - float('inf')
        if mask_data is not None:
            if mask_data.dtype != torch.bool:
                mask_data = mask_data.bool()

            # Segurança: cada linha precisa ter pelo menos uma classe válida
            if not torch.all((~mask_data).any(dim = 1)):
                raise ValueError("Há linhas sem nenhuma classe válida em mask_data.")

            logits = logits.masked_fill(mask_data, -1e9)

    probabilities = torch.softmax(logits, dim = 1)

    if k > 0:
        topk = torch.topk(probabilities, k, dim = 1)
        topk_preds = {'indices': list(topk.indices.detach().numpy()[0]),
                      'probs': list(topk.values.detach().numpy()[0])}
        return topk_preds
    else:
        if tokens_data.shape[0] == 1:
            sorted_item = torch.multinomial(input = probabilities, num_samples = 1).item()
            sorted_prob = probabilities[0, sorted_item].item()
            if extra_info == True:
                return sorted_item, sorted_prob, tokens_data.tolist(), numerical_data.tolist()
            else:
                return sorted_item
        else:
            sorted_items = [torch.multinomial(i, 1).item() for i in probabilities]
            sorted_probs = [probabilities[i, j].item() for i, j in enumerate(sorted_items)]
            if extra_info == True:
                return sorted_items, sorted_probs, tokens_data.tolist(), numerical_data.tolist()
            else:
                return sorted_items

class NGramModel_State(nn.Module):
    """
    Neural Network para N-Gram Language Modeling usando:
      - tokens de contexto
      - um token de estado (ex.: combinação tempo × diferença de pontos)

    Estrutura:
        embedding          : Embedding para os tokens do vocabulário principal.
        state_embedding    : Embedding para o token de estado.
        flatten            : Achata os embeddings dos tokens de contexto.
        network            : MLP sobre [tokens_flattened, state_embed].

    Parâmetros:
        vocab_size (int): Tamanho do vocabulário de tokens.
        context_size (int): Número de tokens de contexto.
        embedding_dim (int): Dimensão do embedding dos tokens.
        state_vocab_size (int): Tamanho do vocabulário de estados (ex.: 225).
        state_embedding_dim (int): Dimensão do embedding de estado.
        num_hidden_neurons (int or list): Número de neurônios em cada camada oculta.
        num_layers (int): Número de camadas ocultas.
        activation (callable): Função de ativação (ex.: nn.GELU, nn.Tanh).
        dropout_rate (float): Probabilidade de dropout.
        use_layer_norm (bool): Se True, aplica LayerNorm em estilo Pre-LN.
    """
    def __init__(
        self,
        vocab_size,
        context_size,
        embedding_dim,
        state_vocab_size,
        state_embedding_dim,
        num_hidden_neurons,
        num_layers,
        activation = nn.Tanh,
        dropout_rate = 0.0,
        use_layer_norm = False,
    ):
        super(NGramModel_State, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.state_embedding = nn.Embedding(state_vocab_size, state_embedding_dim)
        self.flatten = nn.Flatten(start_dim = 1)

        layers = self._init_layers(
            context_size = context_size,
            vocab_size = vocab_size,
            embedding_dim = embedding_dim,
            state_embedding_dim = state_embedding_dim,
            num_hidden_neurons = num_hidden_neurons,
            num_layers = num_layers,
            activation = activation,
            dropout_rate = dropout_rate,
            use_layer_norm = use_layer_norm,
        )

        self.network = nn.Sequential(*layers)

    def _init_layers(
        self,
        context_size,
        vocab_size,
        embedding_dim,
        state_embedding_dim,
        num_hidden_neurons,
        num_layers,
        activation,
        dropout_rate,
        use_layer_norm,
    ):
        layers = []
        # tokens_flattened + state_embedding
        input_size = context_size * embedding_dim + state_embedding_dim

        # num_hidden_neurons pode ser int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada oculta
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas ocultas subsequentes
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Camada de saída
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))
        return layers

    def forward(self, x_tokens, x_state):
        """
        x_tokens : (batch_size, context_size)  índices dos tokens.
        x_state  : (batch_size,)               índices do estado (0..state_vocab_size-1).
        """
        # Embedding dos tokens de contexto
        x = self.embedding(x_tokens)          # (B, C, D)
        x = self.flatten(x)                   # (B, C*D)

        # Embedding do estado
        state_emb = self.state_embedding(x_state)  # (B, state_embedding_dim)

        # Concatena tokens + estado
        x_concat = torch.cat((x, state_emb), dim = 1)

        return self.network(x_concat)
    
def train_state_ngram_model(
    model: torch.nn.Module,
    train_token_data: torch.Tensor,
    train_state_data: torch.Tensor,
    train_targets: torch.Tensor,
    batch_size: int,
    n_epochs: int,
    lr: float = 0.1,
    train_mask_data: Optional[torch.Tensor] = None,
    val_token_data: Optional[torch.Tensor] = None,
    val_state_data: Optional[torch.Tensor] = None,
    val_targets: Optional[torch.Tensor] = None,
    val_mask_data: Optional[torch.Tensor] = None,
    params_optimizer: Optional[dict] = None,
    device: Optional[torch.device] = None,
) -> Tuple[torch.nn.Module, List[List[float]], List[float]]:
    """
    Treina um modelo N-gram que recebe tokens de contexto e um token de estado
    (por exemplo, combinação tempo × diferença de pontos).

    Args:
        model: modelo PyTorch (ex.: NGramModel_State).
        train_token_data: tensor de índices de tokens de treino (N, context_size).
        train_state_data: tensor de índices de estado de treino (N,).
        train_targets: rótulos de treino (N,).
        train_mask_data: máscara de treino (N, vocab_size), onde True indica token proibido (opcional).        
        batch_size: tamanho do batch.
        n_epochs: número de épocas.
        lr: learning rate inicial.
        val_token_data: tokens de validação (opcional).
        val_state_data: estados de validação (opcional).
        val_targets: rótulos de validação (opcional).
        val_mask_data: máscara de validação (N, vocab_size), onde True indica token proibido (opcional).
        params_optimizer: dict com 'factor', 'threshold', 'patience' para o ReduceLROnPlateau.
        device: torch.device("cuda") ou "cpu", se quiser mover tudo.

    Returns:
        model treinado, lista de train_losses (por época, cada um sendo lista por batch),
        lista de val_losses (por época).
    """
    if params_optimizer is None:
        params_optimizer = {'factor': 0.1, 'threshold': 1e-4, 'patience': 1}

    # --------------------------
    # Device
    # --------------------------
    if device is not None:
        model = model.to(device)
        train_token_data = train_token_data.to(device)
        train_state_data = train_state_data.to(device)
        train_targets = train_targets.to(device)

        if train_mask_data is not None:
            train_mask_data = train_mask_data.to(device)

        if (
            val_token_data is not None
            and val_state_data is not None
            and val_targets is not None
        ):
            val_token_data = val_token_data.to(device)
            val_state_data = val_state_data.to(device)
            val_targets = val_targets.to(device)

            if val_mask_data is not None:
                val_mask_data = val_mask_data.to(device)

    # Garantir dtype booleano das máscaras
    if train_mask_data is not None and train_mask_data.dtype != torch.bool:
        train_mask_data = train_mask_data.bool()

    if val_mask_data is not None and val_mask_data.dtype != torch.bool:
        val_mask_data = val_mask_data.bool()

    # --------------------------
    # Otimizador e scheduler
    # --------------------------
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr = lr,
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = params_optimizer['factor'],
        threshold = params_optimizer['threshold'],
        patience = params_optimizer['patience'],
    )

    num_samples = train_token_data.size(0)
    train_losses: List[List[float]] = []
    val_losses: List[float] = []

    best_val_loss = float("inf")
    best_state_dict = None

    # --------------------------
    # Loop de épocas
    # --------------------------
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_losses = []

        # Embaralhar índices
        perm = torch.randperm(num_samples, device = train_token_data.device)

        for idx_start in range(0, num_samples, batch_size):
            idx_end = idx_start + batch_size
            batch_idx = perm[idx_start:idx_end]

            x_tok = train_token_data[batch_idx]   # (B, context_size)
            x_state = train_state_data[batch_idx] # (B,)
            y = train_targets[batch_idx]          # (B,)

            optimizer.zero_grad()
            logits = model(x_tok, x_state)        # chamada específica do NGramModel_State
            if train_mask_data is not None:
                mask_batch = train_mask_data[batch_idx]

                if not torch.all(~mask_batch[torch.arange(y.size(0), device = y.device), y]):
                    raise ValueError("Há targets verdadeiros mascarados no batch de treino.")

                if not torch.all((~mask_batch).any(dim = 1)):
                    raise ValueError("Há linhas sem nenhuma classe válida no batch de treino.")

                logits = logits.masked_fill(mask_batch, -1e9)

            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        train_losses.append(epoch_losses)
        epoch_train_loss = sum(epoch_losses) / len(epoch_losses)

        # --------------------------
        # Validação
        # --------------------------
        if (
            val_token_data is not None
            and val_state_data is not None
            and val_targets is not None
        ):
            model.eval()
            with torch.no_grad():
                val_logits = model(val_token_data, val_state_data)
                if val_mask_data is not None:
                    if not torch.all(~val_mask_data[torch.arange(val_targets.size(0), device = val_targets.device), val_targets]):
                        raise ValueError("Há targets verdadeiros mascarados nos dados de validação.")

                    if not torch.all((~val_mask_data).any(dim = 1)):
                        raise ValueError("Há linhas sem nenhuma classe válida nos dados de validação.")

                    val_logits = val_logits.masked_fill(val_mask_data, -1e9)

                val_loss = F.cross_entropy(val_logits, val_targets).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

            current_lr = optimizer.param_groups[0]['lr']

            # Early stopping baseado no LR mínimo
            if current_lr <= 1e-7:
                if best_state_dict is not None:
                    model.load_state_dict(best_state_dict)
                break

            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f} | "
                f"LR: {current_lr:.2e}",
                end="\r",
                flush=True,
            )
        else:
            print(
                f"[Epoch {epoch:03d}] Train Loss: {epoch_train_loss:.5f}",
                end="\r",
                flush=True,
            )

    # Se teve validação, garantir que o melhor modelo seja carregado
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

def predict_state_ngram_model(
    model,
    tokens_data: torch.Tensor,
    state_data: torch.Tensor,
    k: int = 0,
    extra_info: bool = True,
    mask_data = None
):
    """
    Gera predições a partir de um modelo tipo NGramModel_State,
    que recebe tokens de contexto e um token de estado.

    Args:
        model: modelo PyTorch com forward(x_tokens, x_state).
        tokens_data: tensor de tokens de entrada (B, context_size).
        state_data: tensor com token de estado (B,).
        k: número de top probabilidades a retornar (k>0) ou amostragem (k=0).
        extra_info: se True, retorna também as entradas em formato de lista.
        mask_data (torch.Tensor, optional): Mascara booleana com a mesma dimensao dos logits, 
            onde True indica token proibido.

    Returns:
        Se k > 0:
            dict com {'indices': [...], 'probs': [...]}, top-k da primeira linha.
        Se k == 0 e batch_size == 1:
            (sorted_item, sorted_prob, tokens_list, state_list) se extra_info=True
            sorted_item se extra_info=False
        Se k == 0 e batch_size > 1:
            (sorted_items, sorted_probs, tokens_list, state_list) se extra_info=True
            sorted_items se extra_info=False
    """

    model.eval()
    with torch.no_grad():
        logits = model(tokens_data, state_data)
        logits[:, 0] = -float("inf")
        if mask_data is not None:
            if mask_data.dtype != torch.bool:
                mask_data = mask_data.bool()

            # Segurança: cada linha precisa ter pelo menos uma classe válida
            if not torch.all((~mask_data).any(dim = 1)):
                raise ValueError("Há linhas sem nenhuma classe válida em mask_data.")

            logits = logits.masked_fill(mask_data, -1e9)

    probabilities = torch.softmax(logits, dim=1)

    if k > 0:
        topk = torch.topk(probabilities, k, dim=1)
        topk_preds = {
            "indices": list(topk.indices.detach().cpu().numpy()[0]),
            "probs": list(topk.values.detach().cpu().numpy()[0]),
        }
        return topk_preds
    else:
        batch_size = tokens_data.shape[0]

        if batch_size == 1:
            sorted_item = torch.multinomial(probabilities, num_samples=1).item()
            sorted_prob = probabilities[0, sorted_item].item()

            if extra_info:
                return (
                    sorted_item,
                    sorted_prob,
                    tokens_data.tolist(),
                    state_data.tolist(),
                )
            else:
                return sorted_item
        else:
            sorted_items = [torch.multinomial(p, 1).item() for p in probabilities]
            sorted_probs = [
                probabilities[i, j].item() for i, j in enumerate(sorted_items)
            ]

            if extra_info:
                return (
                    sorted_items,
                    sorted_probs,
                    tokens_data.tolist(),
                    state_data.tolist(),
                )
            else:
                return sorted_items

class NGramModel_ExtraInfo_State(nn.Module):
    """
    Versão do NGramModel_ExtraInfo que inclui um embedding adicional
    para um token de estado (por exemplo, combinação tempo × diferença de pontos).

    Estrutura:
        - Embedding para tokens (sequência de contexto).
        - Embedding para estado (state_id, ex.: 0..224).
        - Flatten dos embeddings de tokens.
        - Concatenação: [tokens_flattened, state_embed, numerical_features]
        - MLP (network) até o output vocab_size.
    """
    def __init__(
        self,
        context_size,
        vocab_size,
        embedding_dim,
        numerical_input_dim,
        state_vocab_size,
        state_embedding_dim,
        num_hidden_neurons,
        num_layers,
        activation = nn.Tanh,
        dropout_rate = 0.0,
        use_layer_norm = False,
    ):
        super(NGramModel_ExtraInfo_State, self).__init__()

        # Embedding dos tokens (jogadas)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        # Embedding do estado (ex.: 225 estados tempo×diferença)
        self.state_embedding = nn.Embedding(state_vocab_size, state_embedding_dim)

        # Inicializa a MLP principal
        layers = self._init_layers(
            context_size = context_size,
            vocab_size = vocab_size,
            embedding_dim = embedding_dim,
            numerical_input_dim = numerical_input_dim,
            state_embedding_dim = state_embedding_dim,
            num_hidden_neurons = num_hidden_neurons,
            num_layers = num_layers,
            activation = activation,
            dropout_rate = dropout_rate,
            use_layer_norm = use_layer_norm,
        )

        self.network = nn.Sequential(*layers)

    def _init_layers(
        self,
        context_size,
        vocab_size,
        embedding_dim,
        numerical_input_dim,
        state_embedding_dim,
        num_hidden_neurons,
        num_layers,
        activation,
        dropout_rate,
        use_layer_norm,
    ):
        """
        Inicializa as camadas da MLP principal, levando em conta:
        - embeddings dos tokens (context_size * embedding_dim)
        - embedding do estado (state_embedding_dim)
        - variáveis numéricas (numerical_input_dim)
        """
        layers = []

        # Tamanho total de input da MLP:
        # [tokens_flattened, state_embed, numerical_features]
        input_size = (
            embedding_dim * context_size
            + state_embedding_dim
            + numerical_input_dim
        )

        # num_hidden_neurons pode ser int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Demais camadas ocultas
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Camada de saída
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))

        return layers

    def forward(self, train_token, train_numerical, train_state):
        """
        Forward pass.

        Parâmetros:
            train_token    : Tensor (batch_size, context_size) com índices dos tokens.
            train_numerical: Tensor (batch_size, numerical_input_dim) com features numéricas.
            train_state    : Tensor (batch_size,) com índices de estado (0..state_vocab_size-1).

        Retorna:
            logits (batch_size, vocab_size)
        """
        # Embedding dos tokens de contexto: (B, C, D)
        embedded_tokens = self.embedding(train_token)
        # Flatten: (B, C*D)
        flattened_tokens = self.flatten(embedded_tokens)

        # Embedding do estado: (B, state_embedding_dim)
        state_emb = self.state_embedding(train_state)

        # Concatena: [tokens_flattened, state_embedding, numerical_features]
        concatenated = torch.cat(
            (flattened_tokens, state_emb, train_numerical),
            dim=1
        )

        # Passa pela MLP principal
        return self.network(concatenated)

def train_num_state_ngram_model(
    model: torch.nn.Module,
    train_token_data: torch.Tensor,
    train_numerical_data: torch.Tensor,
    train_state_data: torch.Tensor,
    train_targets: torch.Tensor,
    batch_size: int,
    n_epochs: int,
    lr: float = 0.1,
    train_mask_data: Optional[torch.Tensor] = None,
    val_token_data: Optional[torch.Tensor] = None,
    val_numerical_data: Optional[torch.Tensor] = None,
    val_state_data: Optional[torch.Tensor] = None,
    val_targets: Optional[torch.Tensor] = None,
    val_mask_data: Optional[torch.Tensor] = None,
    params_optimizer: Optional[dict] = None,
    device: Optional[torch.device] = None,
) -> Tuple[torch.nn.Module, List[List[float]], List[float]]:
    """
    Treina um modelo N-gram com:
      - parte token,
      - parte numérica,
      - e um token de estado (ex.: tempo × diferença de pontos).

    Pensado para modelos do tipo NGramModel_ExtraInfo_State.

    Args:
        model: modelo PyTorch.
        train_token_data: tensor de índices de tokens de treino (N, context_size).
        train_numerical_data: tensor de variáveis numéricas de treino (N, num_features).
        train_state_data: tensor de índices de estado de treino (N,).
        train_targets: rótulos de treino (N,).
        train_mask_data: máscara de treino (N, vocab_size), onde True indica token proibido (opcional).
        batch_size: tamanho do batch.
        n_epochs: número de épocas.
        lr: learning rate inicial.
        val_token_data: tokens de validação (opcional).
        val_numerical_data: numéricos de validação (opcional).
        val_state_data: estados de validação (opcional).
        val_targets: rótulos de validação (opcional).
        val_mask_data: máscara de validação (N, vocab_size), onde True indica token proibido (opcional).
        params_optimizer: dict com 'factor', 'threshold', 'patience'.
        device: torch.device("cuda") ou "cpu", se quiser mover tudo.
        
    Returns:
        model treinado,
        lista de train_losses (por época, cada elemento é a lista de losses por batch),
        lista de val_losses (uma por época).
    """
    if params_optimizer is None:
        params_optimizer = {'factor': 0.1, 'threshold': 1e-4, 'patience': 1}

    # --------------------------
    # Device
    # --------------------------
    if device is not None:
        model = model.to(device)
        train_token_data = train_token_data.to(device)
        train_numerical_data = train_numerical_data.to(device)
        train_state_data = train_state_data.to(device)
        train_targets = train_targets.to(device)

        if train_mask_data is not None:
            train_mask_data = train_mask_data.to(device)

        if (
            val_token_data is not None
            and val_numerical_data is not None
            and val_state_data is not None
            and val_targets is not None
        ):
            val_token_data = val_token_data.to(device)
            val_numerical_data = val_numerical_data.to(device)
            val_state_data = val_state_data.to(device)
            val_targets = val_targets.to(device)

            if val_mask_data is not None:
                val_mask_data = val_mask_data.to(device)

    # Garantir dtype booleano das máscaras
    if train_mask_data is not None and train_mask_data.dtype != torch.bool:
        train_mask_data = train_mask_data.bool()

    if val_mask_data is not None and val_mask_data.dtype != torch.bool:
        val_mask_data = val_mask_data.bool()

    # --------------------------
    # Otimizador e scheduler
    # --------------------------
    optimizer = torch.optim.Adam(
        model.parameters(), lr = lr, weight_decay = 1e-4,)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = params_optimizer['factor'],
        threshold = params_optimizer['threshold'],
        patience = params_optimizer['patience'],
    )

    num_samples = train_token_data.size(0)
    train_losses: List[List[float]] = []
    val_losses: List[float] = []

    best_val_loss = float("inf")
    best_state_dict = None

    # --------------------------
    # Loop de épocas
    # --------------------------
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_losses = []

        # Embaralhar índices
        perm = torch.randperm(num_samples, device=train_token_data.device)

        for idx_start in range(0, num_samples, batch_size):
            idx_end = idx_start + batch_size
            batch_idx = perm[idx_start:idx_end]

            x_tok = train_token_data[batch_idx]        # (B, context_size)
            x_num = train_numerical_data[batch_idx]    # (B, num_num_features)
            x_state = train_state_data[batch_idx]      # (B,)
            y = train_targets[batch_idx]               # (B,)

            optimizer.zero_grad()
            logits = model(x_tok, x_num, x_state)      # <<< assinatura do NGramModel_ExtraInfo_State

            if train_mask_data is not None:
                mask_batch = train_mask_data[batch_idx]

                if not torch.all(~mask_batch[torch.arange(y.size(0), device = y.device), y]):
                    raise ValueError("Há targets verdadeiros mascarados no batch de treino.")

                if not torch.all((~mask_batch).any(dim = 1)):
                    raise ValueError("Há linhas sem nenhuma classe válida no batch de treino.")

                logits = logits.masked_fill(mask_batch, -1e9)
                
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        train_losses.append(epoch_losses)
        epoch_train_loss = sum(epoch_losses) / len(epoch_losses)

        # --------------------------
        # Validação
        # --------------------------
        if (
            val_token_data is not None
            and val_numerical_data is not None
            and val_state_data is not None
            and val_targets is not None
        ):
            model.eval()
            with torch.no_grad():
                val_logits = model(val_token_data, val_numerical_data, val_state_data)
                if val_mask_data is not None:
                    if not torch.all(~val_mask_data[torch.arange(val_targets.size(0), device=val_targets.device), val_targets]):
                        raise ValueError("Há targets verdadeiros mascarados nos dados de validação.")

                    if not torch.all((~val_mask_data).any(dim=1)):
                        raise ValueError("Há linhas sem nenhuma classe válida nos dados de validação.")

                    val_logits = val_logits.masked_fill(val_mask_data, -1e9)              
                val_loss = F.cross_entropy(val_logits, val_targets).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

            current_lr = optimizer.param_groups[0]['lr']

            # Early stopping baseado no LR mínimo
            if current_lr <= 1e-7:
                if best_state_dict is not None:
                    model.load_state_dict(best_state_dict)
                break

            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f} | "
                f"LR: {current_lr:.2e}",
                end = "\r",
                flush = True,
            )
        else:
            print(
                f"[Epoch {epoch:03d}] Train Loss: {epoch_train_loss:.5f}",
                end="\r",
                flush=True,
            )

    # Se teve validação, garantir que o melhor modelo seja carregado
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

def predict_num_state_ngram_model(
    model,
    tokens_data: torch.Tensor,
    numerical_data: torch.Tensor,
    state_data: torch.Tensor,
    k: int = 0,
    extra_info: bool = True,
    mask_data = None
):
    """
    Gera predições a partir de um modelo tipo NGramModel_ExtraInfo_State,
    que recebe tokens de contexto, variáveis numéricas e um token de estado.

    Args:
        model: modelo PyTorch com forward(x_tokens, x_numerical, x_state).
        tokens_data: tensor de tokens de entrada (B, context_size).
        numerical_data: tensor de variáveis numéricas (B, num_features).
        state_data: tensor com token de estado (B,).
        k: número de top probabilidades a retornar (k>0) ou amostragem (k=0).
        extra_info: se True, retorna também as entradas em formato de lista.
        mask_data (torch.Tensor, optional): Mascara booleana com a mesma dimensao dos logits, 
            onde True indica token proibido.

    Returns:
        Igual à função original, mas incluindo state_data no extra_info.
    """

    model.eval()
    with torch.no_grad():
        logits = model(tokens_data, numerical_data, state_data)
        logits[:, 0] = -float("inf")
        if mask_data is not None:
            if mask_data.dtype != torch.bool:
                mask_data = mask_data.bool()

            # Segurança: cada linha precisa ter pelo menos uma classe válida
            if not torch.all((~mask_data).any(dim = 1)):
                raise ValueError("Há linhas sem nenhuma classe válida em mask_data.")

            logits = logits.masked_fill(mask_data, -1e9)

    probabilities = torch.softmax(logits, dim=1)

    if k > 0:
        topk = torch.topk(probabilities, k, dim=1)
        topk_preds = {
            "indices": list(topk.indices.detach().cpu().numpy()[0]),
            "probs": list(topk.values.detach().cpu().numpy()[0]),
        }
        return topk_preds
    else:
        batch_size = tokens_data.shape[0]

        if batch_size == 1:
            sorted_item = torch.multinomial(probabilities, num_samples=1).item()
            sorted_prob = probabilities[0, sorted_item].item()

            if extra_info:
                return (
                    sorted_item,
                    sorted_prob,
                    tokens_data.tolist(),
                    numerical_data.tolist(),
                    state_data.tolist(),
                )
            else:
                return sorted_item
        else:
            sorted_items = [torch.multinomial(p, 1).item() for p in probabilities]
            sorted_probs = [
                probabilities[i, j].item() for i, j in enumerate(sorted_items)
            ]

            if extra_info:
                return (
                    sorted_items,
                    sorted_probs,
                    tokens_data.tolist(),
                    numerical_data.tolist(),
                    state_data.tolist(),
                )
            else:
                return sorted_items

class NGramModel_ExtraInfo_TimeDiffTower(nn.Module):
    """
    Versão estendida da NGramModel_ExtraInfo que inclui:
        - Torre MLP pequena para variáveis de tempo/diferença de pontos
        - Concatenação com embeddings e variáveis numéricas tradicionais

    Estrutura:
        embedding → flatten
        time_diff_features → time_diff_tower
        numerical_features → entram direto
        concat → MLP principal
    """

    def __init__(
        self,
        context_size,
        vocab_size,
        embedding_dim,
        numerical_input_dim,     # variáveis numéricas comuns
        time_diff_input_dim,     # ex: 2 (tempo_restante, diff_pontos)
        time_tower_hidden_dim,   # tamanho da torre
        time_tower_out_dim,      # saída da torre
        num_hidden_neurons,
        num_layers,
        activation = nn.Tanh,
        dropout_rate = 0.0,
        use_layer_norm = False,
    ):
        super().__init__()

        # -------- EMBEDDING + FLATTEN --------
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        # -------- TORRE PARA TEMPO E DIFERENÇA --------
        self.time_tower = nn.Sequential(
            nn.Linear(time_diff_input_dim, time_tower_hidden_dim),
            activation(),
            nn.Dropout(dropout_rate),
            nn.Linear(time_tower_hidden_dim, time_tower_out_dim),
            activation(),
            nn.Dropout(dropout_rate),
        )

        # -------- MLP PRINCIPAL (igual ao ExtraInfo original) --------
        layers = self._init_layers(
            context_size = context_size,
            vocab_size = vocab_size,
            embedding_dim = embedding_dim,
            numerical_input_dim = numerical_input_dim + time_tower_out_dim,
            num_hidden_neurons = num_hidden_neurons,
            num_layers = num_layers,
            activation = activation,
            dropout_rate = dropout_rate,
            use_layer_norm = use_layer_norm,
        )

        self.network = nn.Sequential(*layers)

    def _init_layers(
        self,
        context_size,
        vocab_size,
        embedding_dim,
        numerical_input_dim,
        num_hidden_neurons,
        num_layers,
        activation,
        dropout_rate,
        use_layer_norm,
    ):
        layers = []
        input_size = embedding_dim * context_size + numerical_input_dim

        # num_hidden_neurons pode ser lista ou int
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas seguintes
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Camada de saída
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))

        return layers

    def forward(self, train_token, time_diff_features, numerical_features):
        """
        train_token         -> (B, context_size)
        time_diff_features  -> (B, time_diff_input_dim) ex.: [remaining_time, score_diff]
        numerical_features  -> (B, num_numéricas)
        """

        # Embeddings
        embedded = self.embedding(train_token)
        flattened = self.flatten(embedded)

        # Torre dos estados de tempo e diferença
        time_repr = self.time_tower(time_diff_features)

        # Concat final
        concatenated = torch.cat((flattened, time_repr, numerical_features), dim=1)

        return self.network(concatenated)

def train_time_diff_ngram_model(
    model: torch.nn.Module,
    train_token_data: torch.Tensor,
    train_time_diff_data: torch.Tensor,
    train_numerical_data: torch.Tensor,
    train_targets: torch.Tensor,
    batch_size: int,
    n_epochs: int,
    lr: float = 0.1,
    train_mask_data: Optional[torch.Tensor] = None,
    val_token_data: Optional[torch.Tensor] = None,
    val_time_diff_data: Optional[torch.Tensor] = None,
    val_numerical_data: Optional[torch.Tensor] = None,
    val_targets: Optional[torch.Tensor] = None,
    val_mask_data: Optional[torch.Tensor] = None,
    params_optimizer: Optional[dict] = None,
    device: Optional[torch.device] = None,
) -> Tuple[torch.nn.Module, List[List[float]], List[float]]:
    """
    Treina um NGramModel_ExtraInfo_TimeDiffTower.

    Args:
        model: modelo PyTorch.
        train_token_data: tensor de índices de tokens de treino (N, context_size).
        train_time_diff_data: tensor com variáveis de tempo/diferença (N, time_diff_dim).
        train_numerical_data: tensor com demais variáveis numéricas (N, num_dim).
        train_targets: rótulos de treino (N,).
        train_mask_data: máscara de treino (N, vocab_size), onde True indica token proibido (opcional).        
        batch_size: tamanho do batch.
        n_epochs: número de épocas.
        lr: learning rate inicial.
        val_*: dados de validação (opcionais).
        val_mask_data: máscara de validação (N, vocab_size), onde True indica token proibido (opcional).
        params_optimizer: dict com 'factor', 'threshold', 'patience'.
        device: torch.device("cuda") ou "cpu", se quiser mover tudo.
                
    Returns:
        model treinado,
        train_losses: lista de épocas, cada época contém lista de losses por batch,
        val_losses: lista de losses de validação por época (se houver validação).
    """
    if params_optimizer is None:
        params_optimizer = {'factor': 0.1, 'threshold': 1e-4, 'patience': 1}

    num_samples = train_token_data.size(0)

    # Checagens básicas de consistência
    if train_time_diff_data.size(0) != num_samples:
        raise ValueError("train_time_diff_data deve ter o mesmo número de linhas que train_token_data.")
    if train_numerical_data.size(0) != num_samples:
        raise ValueError("train_numerical_data deve ter o mesmo número de linhas que train_token_data.")
    if train_targets.size(0) != num_samples:
        raise ValueError("train_targets deve ter o mesmo número de linhas que train_token_data.")
    if train_mask_data is not None and train_mask_data.size(0) != num_samples:
        raise ValueError("train_mask_data deve ter o mesmo número de linhas que train_token_data.")
    
    has_val = (
        val_token_data is not None
        and val_time_diff_data is not None
        and val_numerical_data is not None
        and val_targets is not None
    )
    if has_val:
        if val_token_data.size(0) != val_time_diff_data.size(0):
            raise ValueError("val_token_data e val_time_diff_data devem ter o mesmo N.")
        if val_token_data.size(0) != val_numerical_data.size(0):
            raise ValueError("val_token_data e val_numerical_data devem ter o mesmo N.")
        if val_token_data.size(0) != val_targets.size(0):
            raise ValueError("val_token_data e val_targets devem ter o mesmo N.")
        if val_mask_data is not None and val_token_data.size(0) != val_mask_data.size(0):
            raise ValueError("val_mask_data deve ter o mesmo número de linhas que val_token_data.")
    # --------------------------
    # Device
    # --------------------------
    if device is not None:
        model = model.to(device)
        train_token_data     = train_token_data.to(device)
        train_time_diff_data = train_time_diff_data.to(device)
        train_numerical_data = train_numerical_data.to(device)
        train_targets        = train_targets.to(device)

        if train_mask_data is not None:
            train_mask_data = train_mask_data.to(device)

        if has_val:
            val_token_data     = val_token_data.to(device)
            val_time_diff_data = val_time_diff_data.to(device)
            val_numerical_data = val_numerical_data.to(device)
            val_targets        = val_targets.to(device)

            if val_mask_data is not None:
                val_mask_data = val_mask_data.to(device)

    # Garantir dtype booleano das máscaras
    if train_mask_data is not None and train_mask_data.dtype != torch.bool:
        train_mask_data = train_mask_data.bool()

    if val_mask_data is not None and val_mask_data.dtype != torch.bool:
        val_mask_data = val_mask_data.bool()

    # --------------------------
    # Otimizador e scheduler
    # --------------------------
    optimizer = torch.optim.Adam(
        model.parameters(), lr = lr, weight_decay = 1e-4,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = params_optimizer['factor'],
        threshold = params_optimizer['threshold'],
        patience = params_optimizer['patience'],
    )

    train_losses: List[List[float]] = []
    val_losses: List[float] = []
    best_val_loss = float("inf")
    best_state_dict = None

    # --------------------------
    # Loop de épocas
    # --------------------------
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_losses: List[float] = []

        perm = torch.randperm(num_samples, device = train_token_data.device)

        for start in range(0, num_samples, batch_size):
            end = start + batch_size
            batch_idx = perm[start:end]

            x_tok   = train_token_data[batch_idx]
            x_time  = train_time_diff_data[batch_idx]
            x_num   = train_numerical_data[batch_idx]
            y       = train_targets[batch_idx]

            optimizer.zero_grad()
            logits = model(x_tok, x_time, x_num)
            if train_mask_data is not None:
                mask_batch = train_mask_data[batch_idx]

                if not torch.all(~mask_batch[torch.arange(y.size(0), device = y.device), y]):
                    raise ValueError("Há targets verdadeiros mascarados no batch de treino.")

                if not torch.all((~mask_batch).any(dim = 1)):
                    raise ValueError("Há linhas sem nenhuma classe válida no batch de treino.")

                logits = logits.masked_fill(mask_batch, -1e9)

            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        train_losses.append(epoch_losses)
        epoch_train_loss = sum(epoch_losses) / len(epoch_losses)

        # --------------------------
        # Validação
        # --------------------------
        if has_val:
            model.eval()
            with torch.no_grad():
                val_logits = model(val_token_data, val_time_diff_data, val_numerical_data)
                if val_mask_data is not None:
                    if not torch.all(~val_mask_data[torch.arange(val_targets.size(0), device = val_targets.device), val_targets]):
                        raise ValueError("Há targets verdadeiros mascarados nos dados de validação.")

                    if not torch.all((~val_mask_data).any(dim = 1)):
                        raise ValueError("Há linhas sem nenhuma classe válida nos dados de validação.")

                    val_logits = val_logits.masked_fill(val_mask_data, -1e9)

                val_loss = F.cross_entropy(val_logits, val_targets).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

            current_lr = optimizer.param_groups[0]['lr']

            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f} | "
                f"LR: {current_lr:.2e}",
                end = "\r",
                flush = True,
            )

            # Early stopping simples com LR mínimo
            if current_lr <= 1e-7:
                if best_state_dict is not None:
                    model.load_state_dict(best_state_dict)
                break
        else:
            print(
                f"[Epoch {epoch:03d}] Train Loss: {epoch_train_loss:.5f}",
                end = "\r",
                flush = True,
            )

    # Garante carregar o melhor modelo em validação (se houver)
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

def predict_time_diff_ngram_model(
    model: torch.nn.Module,
    tokens_data: torch.Tensor,
    time_diff_data: torch.Tensor,
    numerical_data: torch.Tensor,
    k: int = 0,
    extra_info: bool = True,
    mask_data = None
):
    """
    Gera predições de um NGramModel_ExtraInfo_TimeDiffTower.

    Parâmetros:
        model: modelo já treinado.
        tokens_data: tensor (B, context_size).
        time_diff_data: tensor (B, time_diff_dim).
        numerical_data: tensor (B, num_dim).
        k: se > 0, retorna top-k (da primeira amostra do batch).
        extra_info: se True e k == 0, retorna também os inputs usados.
        mask_data (torch.Tensor, optional): Mascara booleana com a mesma dimensao dos logits, 
            onde True indica token proibido.
    Retorno:
        - Se k > 0:
            dict com 'indices' e 'probs' (da primeira linha).
        - Se k == 0 e batch_size == 1:
            se extra_info: (sorted_item, sorted_prob, tokens_list, time_list, num_list)
            senão: sorted_item
        - Se k == 0 e batch_size > 1:
            se extra_info: (sorted_items, sorted_probs, tokens_list, time_list, num_list)
            senão: sorted_items
    """
    model.eval()
    with torch.no_grad():
        logits = model(tokens_data, time_diff_data, numerical_data)
        logits[:, 0] = -float("inf")
        if mask_data is not None:
            if mask_data.dtype != torch.bool:
                mask_data = mask_data.bool()

            # Segurança: cada linha precisa ter pelo menos uma classe válida
            if not torch.all((~mask_data).any(dim = 1)):
                raise ValueError("Há linhas sem nenhuma classe válida em mask_data.")

            logits = logits.masked_fill(mask_data, -1e9)

        probabilities = torch.softmax(logits, dim = 1)

    # --------------------------
    # Caso top-k
    # --------------------------
    if k > 0:
        topk = torch.topk(probabilities, k, dim = 1)
        topk_preds = {
            'indices': list(topk.indices.detach().cpu().numpy()[0]),
            'probs'  : list(topk.values.detach().cpu().numpy()[0]),
        }
        return topk_preds

    # --------------------------
    # Amostragem multinomial
    # --------------------------
    batch_size = tokens_data.shape[0]

    if batch_size == 1:
        sorted_item = torch.multinomial(input = probabilities, num_samples = 1).item()
        sorted_prob = probabilities[0, sorted_item].item()

        if not extra_info:
            return sorted_item

        tokens_list = tokens_data.tolist()
        time_list   = time_diff_data.tolist()
        num_list    = numerical_data.tolist()

        return sorted_item, sorted_prob, tokens_list, time_list, num_list

    else:
        sorted_items = [torch.multinomial(p, 1).item() for p in probabilities]
        sorted_probs = [probabilities[i, j].item() for i, j in enumerate(sorted_items)]

        if not extra_info:
            return sorted_items

        tokens_list = tokens_data.tolist()
        time_list   = time_diff_data.tolist()
        num_list    = numerical_data.tolist()

        return sorted_items, sorted_probs, tokens_list, time_list, num_list

class NGramModel_ExtraInfo_MultiTowers(nn.Module):
    """
    N-Gram Neural Network com:
      - 1 torre para tokens (embedding + MLP pequena de 2 camadas);
      - N torres para grupos de variáveis numéricas (cada uma MLP de 2 camadas);
      - 1 MLP de fusão em cima das representações.

    Uso:
      - forward(token_ids, numerical_groups)
      - numerical_groups é SEMPRE uma lista (ou tupla) de tensores.
        Ex: [X_time_diff, X_score_context, X_flags]
        Cada tensor com shape (B, d_i).

    Parâmetros:
        context_size (int): Número de tokens de contexto.
        vocab_size (int): Tamanho do vocabulário.
        embedding_dim (int): Dimensão do embedding dos tokens.

        numeric_input_dims (list[int]):
            Dimensão de entrada de cada grupo numérico.
            Ex: [2, 5, 10] → 3 torres com essas dimensões.

        token_tower_dim (int): Dimensão da saída da torre de tokens
                               (também usada como hidden interno).

        numeric_tower_dims (int ou list[int]):
            - int  → mesma dimensão de saída para todas as torres numéricas
                     (também usada como hidden interno).
            - list → dimensão de saída específica por torre.

        num_hidden_neurons (int ou list[int]): Neurônios da MLP de fusão.
        num_layers (int): Número de camadas ocultas da MLP de fusão.
        activation (callable): Função de ativação (ex.: nn.GELU).
        dropout_rate (float): Probabilidade de dropout.
        use_layer_norm (bool): Se True, usa LayerNorm na MLP de fusão (pre-LN).
    """

    def __init__(
        self,
        context_size: int,
        vocab_size: int,
        embedding_dim: int,
        numeric_input_dims: List[int],
        token_tower_dim: int,
        numeric_tower_dims,
        num_hidden_neurons,
        num_layers: int,
        activation = nn.GELU,
        dropout_rate: float = 0.0,
        use_layer_norm: bool = False,
    ):
        super().__init__()

        self.context_size = context_size
        self.embedding_dim = embedding_dim
        self.token_tower_dim = token_tower_dim

        # -------- Normaliza numeric_input_dims --------
        if not isinstance(numeric_input_dims, (list, tuple)) or len(numeric_input_dims) == 0:
            raise ValueError("numeric_input_dims deve ser uma lista não vazia de inteiros.")
        self.numeric_input_dims = list(numeric_input_dims)
        self.num_numeric_towers = len(self.numeric_input_dims)

        # -------- Normaliza numeric_tower_dims --------
        if isinstance(numeric_tower_dims, int):
            self.numeric_tower_dims = [numeric_tower_dims] * self.num_numeric_towers
        else:
            self.numeric_tower_dims = list(numeric_tower_dims)
            if len(self.numeric_tower_dims) != self.num_numeric_towers:
                raise ValueError(
                    "numeric_tower_dims deve ser int ou lista com mesmo comprimento de numeric_input_dims."
                )

        # -------- Torre de tokens (2 camadas) --------
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        token_input_dim = context_size * embedding_dim
        self.token_tower = nn.Sequential(
            nn.Linear(token_input_dim, token_tower_dim),
            activation(),
            nn.Dropout(dropout_rate),
            nn.Linear(token_tower_dim, token_tower_dim),
            activation(),
            nn.Dropout(dropout_rate),
        )

        # -------- Torres numéricas (cada uma com 2 camadas) --------
        self.numeric_towers = nn.ModuleList()
        for in_dim, out_dim in zip(self.numeric_input_dims, self.numeric_tower_dims):
            tower = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                activation(),
                nn.Dropout(dropout_rate),
                nn.Linear(out_dim, out_dim),
                activation(),
                nn.Dropout(dropout_rate),
            )
            self.numeric_towers.append(tower)

        # -------- MLP de fusão --------
        fusion_layers = self._init_fusion_layers(
            vocab_size = vocab_size,
            token_tower_dim = token_tower_dim,
            numeric_tower_dims = self.numeric_tower_dims,
            num_hidden_neurons = num_hidden_neurons,
            num_layers = num_layers,
            activation = activation,
            dropout_rate = dropout_rate,
            use_layer_norm = use_layer_norm,
        )
        self.network = nn.Sequential(*fusion_layers)

    def _init_fusion_layers(
        self,
        vocab_size: int,
        token_tower_dim: int,
        numeric_tower_dims: List[int],
        num_hidden_neurons,
        num_layers: int,
        activation,
        dropout_rate: float,
        use_layer_norm: bool,
    ):
        layers = []
        input_size = token_tower_dim + sum(numeric_tower_dims)

        # num_hidden_neurons pode ser int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        else:
            num_hidden_neurons = list(num_hidden_neurons)
            if len(num_hidden_neurons) != num_layers:
                raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas seguintes
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Saída
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))
        return layers

    def forward(self, token_ids: torch.Tensor, numerical_groups: List[torch.Tensor]):
        """
        token_ids:        (B, context_size)
        numerical_groups: lista de tensores, cada um (B, d_i),
                          com len(numerical_groups) == num_numeric_towers
        """
        if not isinstance(numerical_groups, (list, tuple)):
            raise TypeError("numerical_groups deve ser uma lista/tupla de tensores.")

        if len(numerical_groups) != self.num_numeric_towers:
            raise ValueError(
                f"Foram passados {len(numerical_groups)} grupos numéricos, "
                f"mas o modelo espera {self.num_numeric_towers} torres."
            )

        # Torre de tokens
        embedded = self.embedding(token_ids)          # (B, C, D)
        flattened = self.flatten(embedded)           # (B, C*D)
        token_repr = self.token_tower(flattened)     # (B, token_tower_dim)

        # Torres numéricas
        numeric_reprs = []
        for g, tower in zip(numerical_groups, self.numeric_towers):
            numeric_reprs.append(tower(g))           # cada um (B, out_dim_i)

        # Fusão
        fused = torch.cat([token_repr] + numeric_reprs, dim=1)  # (B, token_dim + sum(num_dims))
        logits = self.network(fused)                            # (B, vocab_size)
        return logits

def train_num_ngram_model_multi_towers(
    model: nn.Module,
    train_token_data: torch.Tensor,
    train_numerical_groups: List[torch.Tensor],
    train_targets: torch.Tensor,
    batch_size: int,
    n_epochs: int,
    lr: float = 0.1,
    train_mask_data: Optional[torch.Tensor] = None,
    val_token_data: Optional[torch.Tensor] = None,
    val_numerical_groups: Optional[List[torch.Tensor]] = None,
    val_targets: Optional[torch.Tensor] = None,
    val_mask_data: Optional[torch.Tensor] = None,
    params_optimizer: Optional[dict] = None,
    device: Optional[torch.device] = None,
) -> Tuple[nn.Module, List[List[float]], List[float]]:
    """
    Treina NGramModel_ExtraInfo_MultiTowers.

    train_numerical_groups: lista de tensores [G1, G2, ..., GK], cada um (N, d_i)
    val_numerical_groups: mesma estrutura, se fornecido.

    Args:
        model: modelo PyTorch.
        train_token_data: tensor de índices de tokens de treino (N, context_size).
        train_numerical_groups: lista de tensores de variáveis numéricas de treino (N, num_features).
        train_state_data: tensor de índices de estado de treino (N,).
        train_targets: rótulos de treino (N,).
        train_mask_data: máscara de treino (N, vocab_size), onde True indica token proibido (opcional).
        batch_size: tamanho do batch.
        n_epochs: número de épocas.
        lr: learning rate inicial.
        val_token_data: tokens de validação (opcional).
        val_numerical_groups: lista de tensortes de variáveis numéricas de validação (opcional).
        val_state_data: estados de validação (opcional).
        val_targets: rótulos de validação (opcional).
        val_mask_data: máscara de validação (N, vocab_size), onde True indica token proibido (opcional).
        params_optimizer: dict com 'factor', 'threshold', 'patience'.
        device: torch.device("cuda") ou "cpu", se quiser mover tudo.

    Retorna:
        model treinado,
        train_losses (list[epoch] -> list[batches]),
        val_losses (list[epoch]) se houver validação.
    """

    if params_optimizer is None:
        params_optimizer = {'factor': 0.1, 'threshold': 1e-4, 'patience': 1}

    # --------------------------
    # Checagens de entrada
    # --------------------------
    if not isinstance(train_numerical_groups, (list, tuple)):
        raise TypeError("train_numerical_groups deve ser uma lista/tupla de tensores.")

    num_groups = len(train_numerical_groups)
    if num_groups == 0:
        raise ValueError("train_numerical_groups não pode ser lista vazia para MultiTowers.")

    num_samples = train_token_data.size(0)

    if train_targets.size(0) != num_samples:
        raise ValueError("train_targets deve ter o mesmo N de train_token_data.")

    for i, g in enumerate(train_numerical_groups):
        if not isinstance(g, torch.Tensor):
            raise TypeError(f"Grupo numérico {i} não é torch.Tensor.")
        if g.size(0) != num_samples:
            raise ValueError("Todos os grupos numéricos devem ter o mesmo N de train_token_data.")
        
    if train_targets.size(0) != num_samples:
        raise ValueError("train_targets deve ter o mesmo N de train_token_data.")

    has_val = (val_token_data is not None) and (val_numerical_groups is not None) and (val_targets is not None)

    if has_val:
        if not isinstance(val_numerical_groups, (list, tuple)):
            raise TypeError("val_numerical_groups deve ser uma lista/tupla de tensores.")
        if len(val_numerical_groups) != num_groups:
            raise ValueError("val_numerical_groups deve ter o mesmo nº de grupos de treino.")

        val_num_samples = val_token_data.size(0)

        if val_targets.size(0) != val_num_samples:
            raise ValueError("val_targets deve ter o mesmo N de val_token_data.")

        for i, g in enumerate(val_numerical_groups):
            if not isinstance(g, torch.Tensor):
                raise TypeError(f"Grupo numérico de validação {i} não é torch.Tensor.")
            if g.size(0) != val_num_samples:
                raise ValueError("Todos os grupos de validação devem ter o mesmo N de val_token_data.")
            
        if val_mask_data is not None and val_mask_data.size(0) != val_num_samples:
            raise ValueError("val_mask_data deve ter o mesmo N de val_token_data.")
        
    # --------------------------
    # Device
    # --------------------------
    if device is not None:
        model = model.to(device)
        train_token_data = train_token_data.to(device)
        train_targets = train_targets.to(device)
        train_numerical_groups = [g.to(device) for g in train_numerical_groups]
        if train_mask_data is not None:
            train_mask_data = train_mask_data.to(device)

        if has_val:
            val_token_data = val_token_data.to(device)
            val_targets = val_targets.to(device)
            val_numerical_groups = [g.to(device) for g in val_numerical_groups]

            if val_mask_data is not None:
                val_mask_data = val_mask_data.to(device)

    # Garantir dtype booleano das máscaras
    if train_mask_data is not None and train_mask_data.dtype != torch.bool:
        train_mask_data = train_mask_data.bool()

    if val_mask_data is not None and val_mask_data.dtype != torch.bool:
        val_mask_data = val_mask_data.bool()
        
    # --------------------------
    # Otimizador e scheduler
    # --------------------------
    optimizer = torch.optim.Adam(
        model.parameters(), lr = lr, weight_decay = 1e-4,)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = params_optimizer['factor'],
        threshold = params_optimizer['threshold'],
        patience = params_optimizer['patience'],
    )

    train_losses: List[List[float]] = []
    val_losses: List[float] = []

    best_val_loss = float("inf")
    best_state_dict = None

    # --------------------------
    # Loop de épocas
    # --------------------------
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_losses: List[float] = []

        perm = torch.randperm(num_samples, device=train_token_data.device)

        for start in range(0, num_samples, batch_size):
            end = start + batch_size
            batch_idx = perm[start:end]

            x_tok = train_token_data[batch_idx]
            y = train_targets[batch_idx]
            batch_groups = [g[batch_idx] for g in train_numerical_groups]

            optimizer.zero_grad(set_to_none = True)
            logits = model(x_tok, batch_groups)
            if train_mask_data is not None:
                mask_batch = train_mask_data[batch_idx]

                if not torch.all(~mask_batch[torch.arange(y.size(0), device = y.device), y]):
                    raise ValueError("Há targets verdadeiros mascarados no batch de treino.")

                if not torch.all((~mask_batch).any(dim = 1)):
                    raise ValueError("Há linhas sem nenhuma classe válida no batch de treino.")

                logits = logits.masked_fill(mask_batch, -1e9)

            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        train_losses.append(epoch_losses)
        epoch_train_loss = sum(epoch_losses) / max(1, len(epoch_losses))

        # --------------------------
        # Validação
        # --------------------------
        if has_val:
            model.eval()
            with torch.no_grad():
                val_logits = model(val_token_data, val_numerical_groups)
                if val_mask_data is not None:
                    if not torch.all(~val_mask_data[torch.arange(val_targets.size(0), device = val_targets.device), val_targets]):
                        raise ValueError("Há targets verdadeiros mascarados nos dados de validação.")

                    if not torch.all((~val_mask_data).any(dim = 1)):
                        raise ValueError("Há linhas sem nenhuma classe válida nos dados de validação.")

                    val_logits = val_logits.masked_fill(val_mask_data, -1e9)

                val_loss = F.cross_entropy(val_logits, val_targets).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

            current_lr = optimizer.param_groups[0]['lr']
            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f} | "
                f"LR: {current_lr:.2e}",
                end="\r",
                flush=True,
            )

            if current_lr <= 1e-7:
                if best_state_dict is not None:
                    model.load_state_dict(best_state_dict)
                break
        else:
            print(
                f"[Epoch {epoch:03d}] Train Loss: {epoch_train_loss:.5f}",
                end="\r",
                flush=True,
            )

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

def predict_num_ngram_model_multi_towers(
    model: nn.Module,
    tokens_data: torch.Tensor,
    numerical_groups: Optional[List[torch.Tensor]] = None,
    k: int = 0,
    extra_info: bool = True,
    mask_data = None
):
    """
    Predição para NGramModel_ExtraInfo_MultiTowers.

    tokens_data:      (B, context_size)
    numerical_groups: lista de tensores [G1, ..., GK], cada (B, d_i)

    Se k > 0 → retorna top-k da primeira linha.
    Se k == 0 → amostra 1 token por linha (multinomial), como na função original.
    """

    model.eval()

    # Checagens de input
    if numerical_groups is None:
        # MultiTowers normalmente exige grupos; só permita "None" se o modelo não tiver torres numéricas.
        if hasattr(model, "num_numeric_towers") and getattr(model, "num_numeric_towers") != 0:
            raise ValueError(
                "numerical_groups=None, mas o modelo MultiTowers espera grupos numéricos. "
                "Passe uma lista [G1, ..., GK]."
            )
        numerical_groups = []
    else:
        if not isinstance(numerical_groups, (list, tuple)):
            raise TypeError("numerical_groups deve ser lista/tupla de tensores ou None.")
        # sanity: batch deve bater
        bsz = tokens_data.size(0)
        for i, g in enumerate(numerical_groups):
            if not isinstance(g, torch.Tensor):
                raise TypeError(f"Grupo numérico {i} não é torch.Tensor.")
            if g.size(0) != bsz:
                raise ValueError("Todos os grupos numéricos devem ter o mesmo batch_size de tokens_data.")

    with torch.no_grad():
        logits = model(tokens_data, numerical_groups)
        logits[:, 0] = -float("inf")
        if mask_data is not None:
            if mask_data.dtype != torch.bool:
                mask_data = mask_data.bool()

            # Segurança: cada linha precisa ter pelo menos uma classe válida
            if not torch.all((~mask_data).any(dim = 1)):
                raise ValueError("Há linhas sem nenhuma classe válida em mask_data.")

            logits = logits.masked_fill(mask_data, -1e9)

        probabilities = torch.softmax(logits, dim = 1)

    # Top-k (primeira linha)
    if k > 0:
        topk = torch.topk(probabilities, k, dim = 1)
        return {
            "indices": list(topk.indices.detach().cpu().numpy()[0]),
            "probs": list(topk.values.detach().cpu().numpy()[0]),
        }

    # Amostragem multinomial (1 por linha)
    batch_size = tokens_data.shape[0]

    if batch_size == 1:
        sampled = torch.multinomial(probabilities, 1).item()
        prob = probabilities[0, sampled].item()

        if not extra_info:
            return sampled

        tokens_list = tokens_data.tolist()
        numerical_list = None if numerical_groups is None else [g.tolist() for g in numerical_groups]
        return sampled, prob, tokens_list, numerical_list

    sampled_items = [torch.multinomial(p, 1).item() for p in probabilities]
    sampled_probs = [probabilities[i, j].item() for i, j in enumerate(sampled_items)]

    if not extra_info:
        return sampled_items

    tokens_list = tokens_data.tolist()
    numerical_list = None if numerical_groups is None else [g.tolist() for g in numerical_groups]
    return sampled_items, sampled_probs, tokens_list, numerical_list

class SpentTimeModel(nn.Module):
    """
    A neural network designed to predict time spent based on contextual token 
    information and numerical inputs, with customizable layers, activation functions, 
    optional Layer Normalization, and Dropout.

    Parameters:
        context_size (int): Number of tokens in the context window.
        vocab_size (int): Size of the vocabulary for the embedding layer.
        time_size (int): The output dimension representing time categories or continuous values.
        embedding_dim (int): Dimensionality of the embedding vectors.
        numerical_input_dim (int): Dimensionality of the numerical input features.
        num_hidden_neurons (int or list): Number of neurons in each hidden layer. 
                                          If a list, specifies neurons for each layer.
        num_layers (int): Number of hidden layers in the network.
        activation_function (callable): Activation function to be used in hidden layers.
        dropout_rate (float): Dropout probability (0.0 means no dropout).
        use_layer_norm (bool): Whether to apply Layer Normalization in the layers.
    """

    def __init__(self, context_size, vocab_size, time_size, embedding_dim, numerical_input_dim,
                 num_hidden_neurons, num_layers, activation_function = nn.Tanh, dropout_rate = 0.0, use_layer_norm = False):
        super(SpentTimeModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim = 1)

        # Building the network dynamically based on the number of layers
        layers = []
        input_features = embedding_dim * context_size + numerical_input_dim

        # Handle num_hidden_neurons as a list or single value
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Append the first layer with optional LayerNorm and Dropout
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_features))
        layers.append(nn.Linear(input_features, num_hidden_neurons[0]))
        layers.append(activation_function())
        layers.append(nn.Dropout(dropout_rate))

        # Append additional hidden layers with optional LayerNorm and Dropout
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation_function())
            layers.append(nn.Dropout(dropout_rate))

        # Output layer to predict the time spent
        layers.append(nn.Linear(num_hidden_neurons[-1], time_size))

        # Combine all layers into a Sequential module
        self.network = nn.Sequential(*layers)

    def forward(self, train_token, train_numerical):
        """
        Defines the forward pass of the model using token indices and numerical features.

        Parameters:
            train_token (torch.Tensor): Tensor of token indices.
            train_numerical (torch.Tensor): Tensor of numerical features.

        Returns:
            torch.Tensor: The output tensor representing predicted time values.
        """
        # Embed and flatten token inputs
        embedded = self.embedding(train_token)
        flattened = self.flatten(embedded)

        # Concatenate embedded tokens with numerical inputs
        concatenated = torch.cat((flattened, train_numerical), dim=1)

        # Pass through the network and return output
        return self.network(concatenated)

def train_spent_time_model(
    model: torch.nn.Module,
    train_token_data: torch.Tensor,
    train_numerical_data: torch.Tensor,
    train_targets: torch.Tensor,
    batch_size: int,
    n_epochs: int,
    lr: float = 0.1,
    train_mask_data: Optional[torch.Tensor] = None,
    val_token_data: Optional[torch.Tensor] = None,
    val_numerical_data: Optional[torch.Tensor] = None,
    val_targets: Optional[torch.Tensor] = None,
    val_mask_data: Optional[torch.Tensor] = None,
    params_optimizer: Optional[dict] = None,
    device: Optional[torch.device] = None,
) -> Tuple[torch.nn.Module, List[List[float]], List[float]]:
    """
    Treina um SpentTimeModel com parte token e parte numérica.

    Args:
        model: modelo a ser treinado.
        train_token_data: tensor de índices de tokens de treino (N, ...).
        train_numerical_data: tensor de variáveis numéricas de treino (N, ...).
        train_targets: rótulos de treino (N,).
        train_mask_data: máscara de treino (N, time_size), onde True indica token proibido (opcional).
        batch_size: tamanho do batch.
        n_epochs: número de épocas.
        lr: learning rate inicial.
        val_token_data: tokens de validação (opcional).
        val_numerical_data: numéricos de validação (opcional).
        val_targets: rótulos de validação (opcional).
        val_mask_data: máscara de validação (N, vocab_size), onde True indica token proibido (opcional).
        params_optimizer: dict com 'factor', 'threshold', 'patience' para o scheduler.
        device: torch.device("cuda") ou "cpu" (opcional).
        
    Returns:
        model treinado,
        train_losses: lista de épocas, cada época contém lista de losses por batch,
        val_losses: lista de losses de validação por época (se houver validação).
    """
    if params_optimizer is None:
        params_optimizer = {'factor': 0.1, 'threshold': 1e-4, 'patience': 1}

    # --------------------------
    # Device
    # --------------------------
    if device is not None:
        model = model.to(device)
        train_token_data = train_token_data.to(device)
        train_numerical_data = train_numerical_data.to(device)
        train_targets = train_targets.to(device)
        if train_mask_data is not None:
            train_mask_data = train_mask_data.to(device)

        if (
            val_token_data is not None
            and val_numerical_data is not None
            and val_targets is not None
        ):
            val_token_data = val_token_data.to(device)
            val_numerical_data = val_numerical_data.to(device)
            val_targets = val_targets.to(device)

            if val_mask_data is not None:
                val_mask_data = val_mask_data.to(device)

    if train_mask_data is not None and train_mask_data.dtype != torch.bool:
        train_mask_data = train_mask_data.bool()

    if val_mask_data is not None and val_mask_data.dtype != torch.bool:
        val_mask_data = val_mask_data.bool()

    # --------------------------
    # Otimizador e scheduler
    # --------------------------
    optimizer = torch.optim.Adam(
        model.parameters(), lr = lr, weight_decay = 1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = params_optimizer['factor'],
        threshold = params_optimizer['threshold'],
        patience = params_optimizer['patience']
    )

    num_samples = train_token_data.size(0)
    train_losses: List[List[float]] = []  # lista de épocas → cada época = lista de batches
    val_losses: List[float] = []

    best_val_loss = float("inf")
    best_state_dict = None

    # --------------------------
    # Loop de épocas
    # --------------------------
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_losses: List[float] = []

        # embaralha índices
        perm = torch.randperm(num_samples, device=train_token_data.device)

        # loop de batches
        for start in range(0, num_samples, batch_size):
            end = start + batch_size
            batch_idx = perm[start:end]

            x_tok = train_token_data[batch_idx]
            x_num = train_numerical_data[batch_idx]
            y = train_targets[batch_idx]

            optimizer.zero_grad()
            logits = model(x_tok, x_num)
            if train_mask_data is not None:
                mask_batch = train_mask_data[batch_idx]

                if not torch.all(~mask_batch[torch.arange(y.size(0), device = y.device), y]):
                    raise ValueError("Há targets verdadeiros mascarados no batch de treino.")

                if not torch.all((~mask_batch).any(dim = 1)):
                    raise ValueError("Há linhas sem nenhuma classe válida no batch de treino.")

                logits = logits.masked_fill(mask_batch, -1e9)

            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        train_losses.append(epoch_losses)
        epoch_train_loss = sum(epoch_losses) / len(epoch_losses)

        # --------------------------
        # Validação 1x por época
        # --------------------------
        if (
            val_token_data is not None
            and val_numerical_data is not None
            and val_targets is not None
        ):
            model.eval()
            with torch.no_grad():
                val_logits = model(val_token_data, val_numerical_data)
                if val_mask_data is not None:
                    if not torch.all(~val_mask_data[torch.arange(val_targets.size(0), device=val_targets.device), val_targets]):
                        raise ValueError("Há targets verdadeiros mascarados nos dados de validação.")

                    if not torch.all((~val_mask_data).any(dim=1)):
                        raise ValueError("Há linhas sem nenhuma classe válida nos dados de validação.")

                    val_logits = val_logits.masked_fill(val_mask_data, -1e9)

                val_loss = F.cross_entropy(val_logits, val_targets).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)

            # salva melhor modelo
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

            current_lr = optimizer.param_groups[0]['lr']

            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f} | "
                f"LR: {current_lr:.2e}",
                end = "\r",
                flush = True,
            )

            # early stopping simples via LR mínimo
            if current_lr <= 1e-7:
                if best_state_dict is not None:
                    model.load_state_dict(best_state_dict)
                break
        else:
            # sem validação, loga só treino se quiser
            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f}",
                end = "\r",
                flush = True,
            )

    # garante que o melhor estado de validação seja carregado
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

def predict_spent_time_model(model, tokens_data, numerical_data, k = 0, mask_data = None):
    """
    Generates predictions from an SpentTimeModel.

    Parameters:
        model (SpentTimeModel): The model to generate predictions from.
        tokens_data (torch.Tensor): Input tokens data tensor.
        numerical_data(torch.Tensor): Input numerical data tensor.
        k (int): The number of top probabilities to return.
        mask_data: (torch.Tensor, optional): Boolean mask with same shape as logits,
            where True indicates forbidden classes.

    Returns:
        dict or int: Top k predictions and their probabilities if top_k is specified; 
        otherwise, the most probable class.
    """
    model.eval()
    with torch.no_grad():
        logits = model(tokens_data, numerical_data)
        if mask_data is not None:
            if mask_data.dtype != torch.bool:
                mask_data = mask_data.bool()

            # Segurança: cada linha precisa ter pelo menos uma classe válida
            if not torch.all((~mask_data).any(dim = 1)):
                raise ValueError("Há linhas sem nenhuma classe válida em mask_data.")

            logits = logits.masked_fill(mask_data, -1e9)

    probabilities = torch.softmax(logits, dim = 1)

    if k:
        topk = torch.topk(probabilities, k, dim = 1)
        topk_preds = {'indices': list(topk.indices.detach().numpy()[0]),
                      'probs': list(topk.values.detach().numpy()[0])}
        return topk_preds
    else:
        if tokens_data.shape[0] == 1:
            return torch.multinomial(probabilities, 1).item()
        else:
            return [torch.multinomial(i, 1).item() for i in probabilities]

class SpentTimeModel_ExtraInfo_State(nn.Module):
    """
    Modelo para prever tempo gasto (ou categorias de tempo) usando:
      - contexto de tokens,
      - variáveis numéricas adicionais,
      - um token de estado (ex.: combinação tempo × diferença de pontos).

    Estrutura:
        embedding         : Embedding para tokens.
        state_embedding   : Embedding para o token de estado.
        flatten           : Achata os embeddings dos tokens.
        network           : MLP sobre [tokens_flat, state_emb, numéricos].

    Parâmetros:
        context_size (int): Número de tokens na janela de contexto.
        vocab_size (int): Tamanho do vocabulário de tokens.
        time_size (int): Dimensão de saída (número de classes de tempo ou 1 se contínuo).
        embedding_dim (int): Dimensão do embedding dos tokens.
        numerical_input_dim (int): Dimensão das variáveis numéricas.
        state_vocab_size (int): Tamanho do vocabulário de estados.
        state_embedding_dim (int): Dimensão do embedding de estado.
        num_hidden_neurons (int ou list): Neurônios em cada camada oculta.
        num_layers (int): Número de camadas ocultas.
        activation_function (callable): Função de ativação (ex.: nn.GELU, nn.Tanh).
        dropout_rate (float): Probabilidade de dropout.
        use_layer_norm (bool): Se True, aplica LayerNorm em estilo pre-LN.
    """

    def __init__(
        self,
        context_size,
        vocab_size,
        time_size,
        embedding_dim,
        numerical_input_dim,
        state_vocab_size,
        state_embedding_dim,
        num_hidden_neurons,
        num_layers,
        activation_function = nn.Tanh,
        dropout_rate = 0.0,
        use_layer_norm = False,
    ):
        super(SpentTimeModel_ExtraInfo_State, self).__init__()

        # Embeddings
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.state_embedding = nn.Embedding(state_vocab_size, state_embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        # Tamanho total de entrada: tokens_flat + state_emb + numéricos
        input_features = (
            embedding_dim * context_size
            + state_embedding_dim
            + numerical_input_dim
        )

        # Tratar num_hidden_neurons como int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        layers = []

        # Primeira camada oculta (pre-LN opcional)
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_features))
        layers.append(nn.Linear(input_features, num_hidden_neurons[0]))
        layers.append(activation_function())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas ocultas subsequentes
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation_function())
            layers.append(nn.Dropout(dropout_rate))

        # Camada de saída para prever o "tempo"
        layers.append(nn.Linear(num_hidden_neurons[-1], time_size))

        self.network = nn.Sequential(*layers)

    def forward(self, train_token, train_numerical, train_state):
        """
        Forward pass do modelo.

        Parâmetros:
            train_token    (Tensor): (batch_size, context_size) índices de tokens.
            train_numerical(Tensor): (batch_size, numerical_input_dim) variáveis numéricas.
            train_state    (Tensor): (batch_size,) índices de estado.

        Retorna:
            Tensor (batch_size, time_size): predição de tempo.
        """
        # Embedding dos tokens
        embedded_tokens = self.embedding(train_token)     # (B, C, D)
        flattened_tokens = self.flatten(embedded_tokens)  # (B, C*D)

        # Embedding do estado
        state_emb = self.state_embedding(train_state)     # (B, state_embedding_dim)

        # Concatena tudo: [tokens_flat, state_emb, numéricos]
        concatenated = torch.cat(
            (flattened_tokens, state_emb, train_numerical),
            dim=1
        )

        return self.network(concatenated)

def train_spent_time_model_extra_info_state(
    model: torch.nn.Module,
    train_token_data: torch.Tensor,
    train_numerical_data: torch.Tensor,
    train_state_data: torch.Tensor,
    train_targets: torch.Tensor,
    batch_size: int,
    n_epochs: int,
    lr: float = 0.1,
    train_mask_data: Optional[torch.Tensor] = None,
    val_token_data: Optional[torch.Tensor] = None,
    val_numerical_data: Optional[torch.Tensor] = None,
    val_state_data: Optional[torch.Tensor] = None,
    val_targets: Optional[torch.Tensor] = None,
    val_mask_data: Optional[torch.Tensor] = None,
    params_optimizer: Optional[dict] = None,
    device: Optional[torch.device] = None,
) -> Tuple[torch.nn.Module, List[List[float]], List[float]]:
    """
    Treina um SpentTimeModel_ExtraInfo_State com:
      - parte token,
      - parte numérica,
      - token de estado.

    Args:
        model: modelo a ser treinado (SpentTimeModel_ExtraInfo_State).
        train_token_data: tensor de índices de tokens de treino (N, context_size).
        train_numerical_data: tensor de variáveis numéricas de treino (N, num_features).
        train_state_data: tensor de índices de estado de treino (N,).
        train_targets: rótulos de treino (N,).
        train_mask_data: máscara de treino (N, time_size), onde True indica token proibido (opcional).
        batch_size: tamanho do batch.
        n_epochs: número de épocas.
        lr: learning rate inicial.
        val_token_data: tokens de validação (opcional).
        val_numerical_data: numéricos de validação (opcional).
        val_state_data: estados de validação (opcional).
        val_targets: rótulos de validação (opcional).
        val_mask_data: máscara de validação (N, vocab_size), onde True indica token proibido (opcional).
        params_optimizer: dict com 'factor', 'threshold', 'patience' para o scheduler.
        device: torch.device("cuda") ou "cpu" (opcional).

    Returns:
        model treinado,
        train_losses: lista de épocas, cada época contém lista de losses por batch,
        val_losses: lista de losses de validação por época (se houver validação).
    """

    if params_optimizer is None:
        params_optimizer = {'factor': 0.1, 'threshold': 1e-4, 'patience': 1}

    # --------------------------
    # Device
    # --------------------------
    if device is not None:
        model = model.to(device)
        train_token_data = train_token_data.to(device)
        train_numerical_data = train_numerical_data.to(device)
        train_state_data = train_state_data.to(device)
        train_targets = train_targets.to(device)

        if train_mask_data is not None:
            train_mask_data = train_mask_data.to(device)

        if (
            val_token_data is not None
            and val_numerical_data is not None
            and val_state_data is not None
            and val_targets is not None
        ):
            val_token_data = val_token_data.to(device)
            val_numerical_data = val_numerical_data.to(device)
            val_state_data = val_state_data.to(device)
            val_targets = val_targets.to(device)

            if val_mask_data is not None:
                val_mask_data = val_mask_data.to(device)

    if train_mask_data is not None and train_mask_data.dtype != torch.bool:
        train_mask_data = train_mask_data.bool()

    if val_mask_data is not None and val_mask_data.dtype != torch.bool:
        val_mask_data = val_mask_data.bool()

    # --------------------------
    # Otimizador e scheduler
    # --------------------------
    optimizer = torch.optim.Adam(
        model.parameters(), lr = lr, weight_decay = 1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = params_optimizer['factor'],
        threshold = params_optimizer['threshold'],
        patience = params_optimizer['patience']
    )

    num_samples = train_token_data.size(0)
    train_losses: List[List[float]] = []
    val_losses: List[float] = []

    best_val_loss = float("inf")
    best_state_dict = None

    # --------------------------
    # Loop de épocas
    # --------------------------
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_losses: List[float] = []

        # embaralha índices
        perm = torch.randperm(num_samples, device=train_token_data.device)

        # loop de batches
        for start in range(0, num_samples, batch_size):
            end = start + batch_size
            batch_idx = perm[start:end]

            x_tok   = train_token_data[batch_idx]
            x_num   = train_numerical_data[batch_idx]
            x_state = train_state_data[batch_idx]
            y       = train_targets[batch_idx]

            optimizer.zero_grad()
            logits = model(x_tok, x_num, x_state)
            if train_mask_data is not None:
                mask_batch = train_mask_data[batch_idx]

                if not torch.all(~mask_batch[torch.arange(y.size(0), device=y.device), y]):
                    raise ValueError("Há targets verdadeiros mascarados no batch de treino.")

                if not torch.all((~mask_batch).any(dim=1)):
                    raise ValueError("Há linhas sem nenhuma classe válida no batch de treino.")

                logits = logits.masked_fill(mask_batch, -1e9)            
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        train_losses.append(epoch_losses)
        epoch_train_loss = sum(epoch_losses) / len(epoch_losses)

        # --------------------------
        # Validação 1x por época
        # --------------------------
        if (
            val_token_data is not None
            and val_numerical_data is not None
            and val_state_data is not None
            and val_targets is not None
        ):
            model.eval()
            with torch.no_grad():
                val_logits = model(val_token_data, val_numerical_data, val_state_data)
                if val_mask_data is not None:
                    if not torch.all(~val_mask_data[torch.arange(val_targets.size(0), device = val_targets.device), val_targets]):
                        raise ValueError("Há targets verdadeiros mascarados nos dados de validação.")

                    if not torch.all((~val_mask_data).any(dim = 1)):
                        raise ValueError("Há linhas sem nenhuma classe válida nos dados de validação.")

                    val_logits = val_logits.masked_fill(val_mask_data, -1e9)                
                val_loss = F.cross_entropy(val_logits, val_targets).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)

            # salva melhor modelo
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

            current_lr = optimizer.param_groups[0]['lr']

            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f} | "
                f"LR: {current_lr:.2e}",
                end = "\r",
                flush = True,
            )

            # early stopping simples via LR mínimo
            if current_lr <= 1e-7:
                if best_state_dict is not None:
                    model.load_state_dict(best_state_dict)
                break
        else:
            # sem validação, loga só treino se quiser
            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f}",
                end = "\r",
                flush = True,
            )

    # garante que o melhor estado de validação seja carregado
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

def predict_spent_time_model_extra_info_state(
    model,
    tokens_data: torch.Tensor,
    numerical_data: torch.Tensor,
    state_data: torch.Tensor,
    k: int = 0,
    mask_data = None
):
    """
    Gera predições a partir de um SpentTimeModel_ExtraInfo_State.

    Parâmetros:
        model: modelo SpentTimeModel_ExtraInfo_State.
        tokens_data (torch.Tensor): tensor de tokens de entrada (B, context_size).
        numerical_data(torch.Tensor): tensor de variáveis numéricas (B, num_features).
        state_data (torch.Tensor): tensor com índices de estado (B,).
        k (int): número de top probabilidades a retornar.
        mask_data (torch.Tensor, optional): Mascara booleana com a mesma dimensao dos logits, 
            onde True indica token proibido.

    Returns:
        Se k > 0:
            dict com 'indices' e 'probs' (top-k da primeira linha).
        Se k == 0 e batch_size == 1:
            um único índice amostrado.
        Se k == 0 e batch_size > 1:
            lista de índices amostrados, um por linha do batch.
    """
    model.eval()
    with torch.no_grad():
        logits = model(tokens_data, numerical_data, state_data)
        if mask_data is not None:
            if mask_data.dtype != torch.bool:
                mask_data = mask_data.bool()

            # Segurança: cada linha precisa ter pelo menos uma classe válida
            if not torch.all((~mask_data).any(dim = 1)):
                raise ValueError("Há linhas sem nenhuma classe válida em mask_data.")

            logits = logits.masked_fill(mask_data, -1e9)

    probabilities = torch.softmax(logits, dim = 1)

    if k:
        topk = torch.topk(probabilities, k, dim = 1)
        topk_preds = {
            'indices': list(topk.indices.detach().cpu().numpy()[0]),
            'probs'  : list(topk.values.detach().cpu().numpy()[0]),
        }
        return topk_preds
    else:
        if tokens_data.shape[0] == 1:
            return torch.multinomial(probabilities, 1).item()
        else:
            return [torch.multinomial(p, 1).item() for p in probabilities]

class SpentTimeModel_TimeDiffTower(nn.Module):
    """
    Versão estendida da SpentTimeModel que inclui:
      - torre MLP pequena para variáveis de tempo/diferença (ex: remaining_time, score_diff)
      - concatenação dessa representação com:
          * embeddings dos tokens
          * variáveis numéricas tradicionais

    Estrutura:
        train_token → embedding → flatten
        time_diff_features → time_tower
        train_numerical → entra direto
        concat = [flattened, time_repr, train_numerical]
        → network (MLP principal) → time_size (classes/saídas de tempo)
    """

    def __init__(
        self,
        context_size: int,
        vocab_size: int,
        time_size: int,
        embedding_dim: int,
        numerical_input_dim: int,      # variáveis numéricas comuns
        time_diff_input_dim: int,      # ex: 2 (remaining_time, score_diff)
        time_tower_hidden_dim: int,
        time_tower_out_dim: int,
        num_hidden_neurons,
        num_layers: int,
        activation_function = nn.Tanh,
        dropout_rate: float = 0.0,
        use_layer_norm: bool = False,
    ):
        super(SpentTimeModel_TimeDiffTower, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim = 1)

        # -------- Torre para tempo/diferença --------
        # (sem LayerNorm aqui, como combinamos)
        self.time_tower = nn.Sequential(
            nn.Linear(time_diff_input_dim, time_tower_hidden_dim),
            activation_function(),
            nn.Dropout(dropout_rate),
            nn.Linear(time_tower_hidden_dim, time_tower_out_dim),
            activation_function(),
            nn.Dropout(dropout_rate),
        )

        # -------- MLP principal (igual à SpentTimeModel, mas com input aumentado) --------
        # input_features = embedding*context + numéricas “normais” + representação de tempo
        input_features = embedding_dim * context_size + numerical_input_dim + time_tower_out_dim

        layers = []
        # num_hidden_neurons pode ser int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_features))
        layers.append(nn.Linear(input_features, num_hidden_neurons[0]))
        layers.append(activation_function())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas ocultas adicionais
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation_function())
            layers.append(nn.Dropout(dropout_rate))

        # Saída para tempo (classes / bins de tempo)
        layers.append(nn.Linear(num_hidden_neurons[-1], time_size))

        self.network = nn.Sequential(*layers)

    def forward(
        self,
        train_token: torch.Tensor,         # (B, context_size)
        time_diff_features: torch.Tensor,  # (B, time_diff_input_dim)
        train_numerical: torch.Tensor,     # (B, numerical_input_dim)
    ) -> torch.Tensor:
        """
        Forward pass usando:
          - tokens
          - features de tempo/diferença
          - demais variáveis numéricas
        """

        # Embedding dos tokens
        embedded = self.embedding(train_token)
        flattened = self.flatten(embedded)

        # Torre de tempo/diferença
        time_repr = self.time_tower(time_diff_features)

        # Concatena tudo
        concatenated = torch.cat((flattened, time_repr, train_numerical), dim = 1)

        # MLP principal
        return self.network(concatenated)
    
def train_spent_time_time_diff_model(
    model: torch.nn.Module,
    train_token_data: torch.Tensor,
    train_time_diff_data: torch.Tensor,
    train_numerical_data: torch.Tensor,
    train_targets: torch.Tensor,
    batch_size: int,
    n_epochs: int,
    lr: float = 0.1,
    train_mask_data: Optional[torch.Tensor] = None,
    val_token_data: Optional[torch.Tensor] = None,
    val_time_diff_data: Optional[torch.Tensor] = None,
    val_numerical_data: Optional[torch.Tensor] = None,
    val_targets: Optional[torch.Tensor] = None,
    val_mask_data: Optional[torch.Tensor] = None,
    params_optimizer: Optional[dict] = None,
    device: Optional[torch.device] = None,
) -> Tuple[torch.nn.Module, List[List[float]], List[float]]:
    """
    Treina um SpentTimeModel_TimeDiffTower com:
      - tokens
      - torre de tempo/diferença (time_diff_data)
      - variáveis numéricas comuns
    """

    if params_optimizer is None:
        params_optimizer = {'factor': 0.1, 'threshold': 1e-4, 'patience': 1}

    num_samples = train_token_data.size(0)

    # Checagens
    if train_time_diff_data.size(0) != num_samples:
        raise ValueError("train_time_diff_data deve ter o mesmo N de train_token_data.")
    if train_numerical_data.size(0) != num_samples:
        raise ValueError("train_numerical_data deve ter o mesmo N de train_token_data.")
    if train_targets.size(0) != num_samples:
        raise ValueError("train_targets deve ter o mesmo N de train_token_data.")
    if train_mask_data is not None and train_mask_data.size(0) != num_samples:
        raise ValueError("train_mask_data deve ter o mesmo N de train_token_data.")

    has_val = (
        val_token_data is not None
        and val_time_diff_data is not None
        and val_numerical_data is not None
        and val_targets is not None
    )
    if has_val:
        if val_token_data.size(0) != val_time_diff_data.size(0):
            raise ValueError("val_token_data e val_time_diff_data devem ter o mesmo N.")
        if val_token_data.size(0) != val_numerical_data.size(0):
            raise ValueError("val_token_data e val_numerical_data devem ter o mesmo N.")
        if val_token_data.size(0) != val_targets.size(0):
            raise ValueError("val_token_data e val_targets devem ter o mesmo N.")
        if val_mask_data is not None and val_mask_data.size(0) != val_token_data.size(0):
            raise ValueError("val_mask_data deve ter o mesmo N de val_token_data.")
        
    # Device
    if device is not None:
        model = model.to(device)
        train_token_data     = train_token_data.to(device)
        train_time_diff_data = train_time_diff_data.to(device)
        train_numerical_data = train_numerical_data.to(device)
        train_targets        = train_targets.to(device)
        if train_mask_data is not None:
            train_mask_data = train_mask_data.to(device)

        if has_val:
            val_token_data     = val_token_data.to(device)
            val_time_diff_data = val_time_diff_data.to(device)
            val_numerical_data = val_numerical_data.to(device)
            val_targets        = val_targets.to(device)

            if val_mask_data is not None:
                val_mask_data = val_mask_data.to(device)

    if train_mask_data is not None and train_mask_data.dtype != torch.bool:
        train_mask_data = train_mask_data.bool()

    if val_mask_data is not None and val_mask_data.dtype != torch.bool:
        val_mask_data = val_mask_data.bool()
        
    # Otimizador e scheduler
    optimizer = torch.optim.Adam(
        model.parameters(), lr = lr, weight_decay = 1e-4,)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = params_optimizer['factor'],
        threshold = params_optimizer['threshold'],
        patience = params_optimizer['patience'],
    )

    train_losses: List[List[float]] = []
    val_losses: List[float] = []
    best_val_loss = float("inf")
    best_state_dict = None

    # Loop de épocas
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_losses: List[float] = []

        perm = torch.randperm(num_samples, device = train_token_data.device)

        for start in range(0, num_samples, batch_size):
            end = start + batch_size
            batch_idx = perm[start:end]

            x_tok   = train_token_data[batch_idx]
            x_time  = train_time_diff_data[batch_idx]
            x_num   = train_numerical_data[batch_idx]
            y       = train_targets[batch_idx]

            optimizer.zero_grad()
            logits = model(x_tok, x_time, x_num)
            if train_mask_data is not None:
                mask_batch = train_mask_data[batch_idx]

                if not torch.all(~mask_batch[torch.arange(y.size(0), device = y.device), y]):
                    raise ValueError("Há targets verdadeiros mascarados no batch de treino.")

                if not torch.all((~mask_batch).any(dim = 1)):
                    raise ValueError("Há linhas sem nenhuma classe válida no batch de treino.")

                logits = logits.masked_fill(mask_batch, -1e9)

            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        train_losses.append(epoch_losses)
        epoch_train_loss = sum(epoch_losses) / len(epoch_losses)

        # Validação
        if has_val:
            model.eval()
            with torch.no_grad():
                val_logits = model(val_token_data, val_time_diff_data, val_numerical_data)
                if val_mask_data is not None:
                    if not torch.all(~val_mask_data[torch.arange(val_targets.size(0), device=val_targets.device), val_targets]):
                        raise ValueError("Há targets verdadeiros mascarados nos dados de validação.")

                    if not torch.all((~val_mask_data).any(dim=1)):
                        raise ValueError("Há linhas sem nenhuma classe válida nos dados de validação.")

                    val_logits = val_logits.masked_fill(val_mask_data, -1e9)                
                val_loss = F.cross_entropy(val_logits, val_targets).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

            current_lr = optimizer.param_groups[0]['lr']
            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f} | "
                f"LR: {current_lr:.2e}",
                end = "\r",
                flush = True,
            )

            if current_lr <= 1e-7:
                if best_state_dict is not None:
                    model.load_state_dict(best_state_dict)
                break
        else:
            print(
                f"[Epoch {epoch:03d}] Train Loss: {epoch_train_loss:.5f}",
                end = "\r",
                flush = True,
            )

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

def predict_spent_time_time_diff_model(
    model: torch.nn.Module,
    tokens_data: torch.Tensor,
    time_diff_data: torch.Tensor,
    numerical_data: torch.Tensor,
    k: int = 0,
    mask_data = None
):
    """
    Gera predições de um SpentTimeModel_TimeDiffTower.

    Parâmetros:
        model: SpentTimeModel_TimeDiffTower treinado.
        tokens_data: Tensor (B, context_size).
        time_diff_data: Tensor (B, time_diff_dim).
        numerical_data: Tensor (B, num_dim).
        k: top-k para retornar. Se 0, faz amostragem de uma classe por linha.
        mask_data (torch.Tensor, optional): Mascara booleana com a mesma dimensao dos logits, 
            onde True indica token proibido.

    Retorno:
        Se k > 0:
            dict com 'indices' e 'probs' (da primeira amostra).
        Se k == 0 e B == 1:
            inteiro (classe amostrada).
        Se k == 0 e B > 1:
            lista de inteiros (classes amostradas por linha).
    """
    model.eval()
    with torch.no_grad():
        logits = model(tokens_data, time_diff_data, numerical_data)
        if mask_data is not None:
            if mask_data.dtype != torch.bool:
                mask_data = mask_data.bool()

            # Segurança: cada linha precisa ter pelo menos uma classe válida
            if not torch.all((~mask_data).any(dim = 1)):
                raise ValueError("Há linhas sem nenhuma classe válida em mask_data.")

            logits = logits.masked_fill(mask_data, -1e9)

        probabilities = torch.softmax(logits, dim = 1)

    if k:
        topk = torch.topk(probabilities, k, dim = 1)
        topk_preds = {
            'indices': list(topk.indices.detach().cpu().numpy()[0]),
            'probs'  : list(topk.values.detach().cpu().numpy()[0]),
        }
        return topk_preds
    else:
        if tokens_data.shape[0] == 1:
            return torch.multinomial(probabilities, 1).item()
        else:
            return [torch.multinomial(p, 1).item() for p in probabilities]

class SpentTimeModel_MultiTowers(nn.Module):
    """
    Modelo de tempo gasto com múltiplas torres numéricas (MultiTowers),
    prevendo time_size.

    Atualização: torres (tokens e numéricas) agora têm DUAS camadas,
    no padrão do seu TimeDiffTower:
      Linear(in -> tower_dim) -> act -> drop -> Linear(tower_dim -> tower_dim) -> act -> drop
    """

    def __init__(
        self,
        context_size: int,
        vocab_size: int,
        time_size: int,
        embedding_dim: int,
        numeric_input_dims: List[int],
        token_tower_dim: int,
        numeric_tower_dims,
        num_hidden_neurons,
        num_layers: int,
        activation_function = nn.GELU,
        dropout_rate: float = 0.0,
        use_layer_norm: bool = False,
    ):
        super().__init__()

        self.context_size = context_size
        self.embedding_dim = embedding_dim
        self.token_tower_dim = token_tower_dim
        self.time_size = time_size

        # -------- Normalização das dims numéricas --------
        if not isinstance(numeric_input_dims, (list, tuple)) or len(numeric_input_dims) == 0:
            raise ValueError("numeric_input_dims deve ser uma lista não vazia de inteiros.")
        self.numeric_input_dims = list(numeric_input_dims)
        self.num_numeric_towers = len(self.numeric_input_dims)

        # numeric_tower_dims: int ou lista
        if isinstance(numeric_tower_dims, int):
            self.numeric_tower_dims = [numeric_tower_dims] * self.num_numeric_towers
        else:
            self.numeric_tower_dims = list(numeric_tower_dims)
            if len(self.numeric_tower_dims) != self.num_numeric_towers:
                raise ValueError(
                    "numeric_tower_dims deve ser int ou lista com mesmo comprimento de numeric_input_dims."
                )

        # -------- Torre de tokens --------
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        token_input_dim = context_size * embedding_dim

        self.token_tower = nn.Sequential(
            nn.Linear(token_input_dim, token_tower_dim),
            activation_function(),
            nn.Dropout(dropout_rate),
            nn.Linear(token_tower_dim, token_tower_dim),
            activation_function(),
            nn.Dropout(dropout_rate),
        )

        # -------- Torres numéricas --------
        self.numeric_towers = nn.ModuleList()
        for in_dim, out_dim in zip(self.numeric_input_dims, self.numeric_tower_dims):
            tower = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                activation_function(),
                nn.Dropout(dropout_rate),
                nn.Linear(out_dim, out_dim),
                activation_function(),
                nn.Dropout(dropout_rate),
            )
            self.numeric_towers.append(tower)

        # -------- MLP de fusão --------
        fusion_layers = self._init_fusion_layers(
            time_size=time_size,
            token_tower_dim=token_tower_dim,
            numeric_tower_dims=self.numeric_tower_dims,
            num_hidden_neurons=num_hidden_neurons,
            num_layers=num_layers,
            activation_function=activation_function,
            dropout_rate=dropout_rate,
            use_layer_norm=use_layer_norm,
        )
        self.network = nn.Sequential(*fusion_layers)

    def _init_fusion_layers(
        self,
        time_size: int,
        token_tower_dim: int,
        numeric_tower_dims: List[int],
        num_hidden_neurons,
        num_layers: int,
        activation_function,
        dropout_rate: float,
        use_layer_norm: bool,
    ):
        layers = []
        input_size = token_tower_dim + sum(numeric_tower_dims)

        # num_hidden_neurons: int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        else:
            num_hidden_neurons = list(num_hidden_neurons)
            if len(num_hidden_neurons) != num_layers:
                raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation_function())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas seguintes
        for i in range(1, num_layers):
            in_dim = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation_function())
            layers.append(nn.Dropout(dropout_rate))

        # Saída
        layers.append(nn.Linear(num_hidden_neurons[-1], time_size))
        return layers

    def forward(self, token_ids: torch.Tensor, numerical_groups: List[torch.Tensor]):
        if not isinstance(numerical_groups, (list, tuple)):
            raise TypeError("numerical_groups deve ser uma lista/tupla de tensores.")

        if len(numerical_groups) != self.num_numeric_towers:
            raise ValueError(
                f"Foram passados {len(numerical_groups)} grupos numéricos, "
                f"mas o modelo espera {self.num_numeric_towers} torres."
            )

        embedded = self.embedding(token_ids)          # (B, C, D)
        flattened = self.flatten(embedded)           # (B, C*D)
        token_repr = self.token_tower(flattened)     # (B, token_tower_dim)

        numeric_reprs = []
        for g, tower in zip(numerical_groups, self.numeric_towers):
            numeric_reprs.append(tower(g))           # (B, out_dim_i)

        fused = torch.cat([token_repr] + numeric_reprs, dim=1)  # (B, token_dim + sum(out_dims))
        logits = self.network(fused)                              # (B, time_size)
        return logits
        
def train_spent_time_model_multi_towers(
    model: nn.Module,
    train_token_data: torch.Tensor,
    train_numerical_groups: List[torch.Tensor],
    train_targets: torch.Tensor,
    batch_size: int,
    n_epochs: int,
    lr: float = 0.1,
    train_mask_data: Optional[torch.Tensor] = None,
    val_token_data: Optional[torch.Tensor] = None,
    val_numerical_groups: Optional[List[torch.Tensor]] = None,
    val_targets: Optional[torch.Tensor] = None,
    val_mask_data: Optional[torch.Tensor] = None,
    params_optimizer: Optional[dict] = None,
    device: Optional[torch.device] = None,
) -> Tuple[nn.Module, List[List[float]], List[float]]:
    """
    Treina SpentTimeModel_MultiTowers.

    train_numerical_groups: lista de tensores [G1, ..., GK], cada um (N, d_i)
    val_numerical_groups: mesma estrutura, se fornecido.

    Retorna:
        model treinado,
        train_losses: lista de épocas, cada época contém lista de losses por batch,
        val_losses: lista de losses de validação por época (se houver).
    """
    if params_optimizer is None:
        params_optimizer = {'factor': 0.1, 'threshold': 1e-4, 'patience': 1}

    # --------------------------
    # Checagens básicas
    # --------------------------
    if not isinstance(train_numerical_groups, (list, tuple)):
        raise TypeError("train_numerical_groups deve ser lista/tupla de tensores.")
    num_groups = len(train_numerical_groups)
    if num_groups == 0:
        raise ValueError("train_numerical_groups não pode ser lista vazia.")

    num_samples = train_token_data.size(0)
    if train_targets.size(0) != num_samples:
        raise ValueError("train_targets deve ter o mesmo N de train_token_data.")

    for gi, g in enumerate(train_numerical_groups):
        if g.size(0) != num_samples:
            raise ValueError(f"Grupo numérico {gi} tem N diferente de train_token_data.")
        
    if train_mask_data is not None and train_mask_data.size(0) != num_samples:
        raise ValueError("train_mask_data deve ter o mesmo N de train_token_data.")

    has_val = (
        val_token_data is not None
        and val_numerical_groups is not None
        and val_targets is not None
    )

    if has_val:
        if not isinstance(val_numerical_groups, (list, tuple)):
            raise TypeError("val_numerical_groups deve ser lista/tupla de tensores.")
        if len(val_numerical_groups) != num_groups:
            raise ValueError("val_numerical_groups deve ter o mesmo nº de grupos de treino.")

        if val_token_data.size(0) != val_targets.size(0):
            raise ValueError("val_token_data e val_targets devem ter o mesmo N.")

        for gi, g in enumerate(val_numerical_groups):
            if g.size(0) != val_token_data.size(0):
                raise ValueError(f"Grupo val {gi} tem N diferente de val_token_data.")
            
        if val_mask_data is not None and val_mask_data.size(0) != val_token_data.size(0):
            raise ValueError("val_mask_data deve ter o mesmo N de val_token_data.")            


    # --------------------------
    # Device
    # --------------------------
    if device is not None:
        model = model.to(device)
        train_token_data = train_token_data.to(device)
        train_targets = train_targets.to(device)
        train_numerical_groups = [g.to(device) for g in train_numerical_groups]
        if train_mask_data is not None:
            train_mask_data = train_mask_data.to(device)

        if has_val:
            val_token_data = val_token_data.to(device)
            val_targets = val_targets.to(device)
            val_numerical_groups = [g.to(device) for g in val_numerical_groups]
            if val_mask_data is not None:
                val_mask_data = val_mask_data.to(device)

    if train_mask_data is not None and train_mask_data.dtype != torch.bool:
        train_mask_data = train_mask_data.bool()

    if val_mask_data is not None and val_mask_data.dtype != torch.bool:
        val_mask_data = val_mask_data.bool()

    # --------------------------
    # Otimizador e scheduler
    # --------------------------
    optimizer = torch.optim.Adam(
        model.parameters(), lr = lr, weight_decay = 1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = params_optimizer['factor'],
        threshold = params_optimizer['threshold'],
        patience = params_optimizer['patience']
    )

    train_losses: List[List[float]] = []
    val_losses: List[float] = []

    best_val_loss = float("inf")
    best_state_dict = None

    # --------------------------
    # Loop de épocas
    # --------------------------
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_losses: List[float] = []

        perm = torch.randperm(num_samples, device=train_token_data.device)

        for start in range(0, num_samples, batch_size):
            end = start + batch_size
            batch_idx = perm[start:end]

            x_tok = train_token_data[batch_idx]
            y = train_targets[batch_idx]
            batch_groups = [g[batch_idx] for g in train_numerical_groups]

            optimizer.zero_grad()
            logits = model(x_tok, batch_groups)
            if train_mask_data is not None:
                mask_batch = train_mask_data[batch_idx]

                if not torch.all(~mask_batch[torch.arange(y.size(0), device = y.device), y]):
                    raise ValueError("Há targets verdadeiros mascarados no batch de treino.")

                if not torch.all((~mask_batch).any(dim = 1)):
                    raise ValueError("Há linhas sem nenhuma classe válida no batch de treino.")

                logits = logits.masked_fill(mask_batch, -1e9)            
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        train_losses.append(epoch_losses)
        epoch_train_loss = sum(epoch_losses) / max(1, len(epoch_losses))

        # --------------------------
        # Validação
        # --------------------------
        if has_val:
            model.eval()
            with torch.no_grad():
                val_logits = model(val_token_data, val_numerical_groups)
                if val_mask_data is not None:
                    if not torch.all(~val_mask_data[torch.arange(val_targets.size(0), device = val_targets.device), val_targets]):
                        raise ValueError("Há targets verdadeiros mascarados nos dados de validação.")

                    if not torch.all((~val_mask_data).any(dim = 1)):
                        raise ValueError("Há linhas sem nenhuma classe válida nos dados de validação.")

                    val_logits = val_logits.masked_fill(val_mask_data, -1e9)                
                val_loss = F.cross_entropy(val_logits, val_targets).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

            current_lr = optimizer.param_groups[0]['lr']

            print(
                f"[Epoch {epoch:03d}] "
                f"Train Loss: {epoch_train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f} | "
                f"LR: {current_lr:.2e}",
                end="\r",
                flush=True,
            )

            # early stopping simples via LR mínimo
            if current_lr <= 1e-7:
                if best_state_dict is not None:
                    model.load_state_dict(best_state_dict)
                break
        else:
            print(
                f"[Epoch {epoch:03d}] Train Loss: {epoch_train_loss:.5f}",
                end="\r",
                flush=True,
            )

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, train_losses, val_losses

def predict_spent_time_model_multi_towers(
    model: nn.Module,
    tokens_data: torch.Tensor,
    numerical_groups: List[torch.Tensor],
    k: int = 0,
    mask_data = None
):
    """
    Predição para SpentTimeModel_MultiTowers.

    - Se k > 0: retorna top-k (da 1ª linha) com probs.
    - Se k == 0:
        - se batch==1: amostra 1 classe
        - se batch>1: amostra 1 classe por linha
    """
    model.eval()

    if not isinstance(numerical_groups, (list, tuple)):
        raise TypeError("numerical_groups deve ser lista/tupla de tensores.")
    if len(numerical_groups) == 0:
        raise ValueError("numerical_groups não pode ser lista vazia.")

    # mover tudo pro device do modelo
    model_device = next(model.parameters()).device
    tokens_data = tokens_data.to(model_device)
    numerical_groups = [g.to(model_device) for g in numerical_groups]

    model.eval()
    with torch.no_grad():
        logits = model(tokens_data, numerical_groups)
        if mask_data is not None:
            if mask_data.dtype != torch.bool:
                mask_data = mask_data.bool()

            # Segurança: cada linha precisa ter pelo menos uma classe válida
            if not torch.all((~mask_data).any(dim = 1)):
                raise ValueError("Há linhas sem nenhuma classe válida em mask_data.")

            logits = logits.masked_fill(mask_data, -1e9)

        probabilities = torch.softmax(logits, dim = 1)

    if k and k > 0:
        topk = torch.topk(probabilities, k, dim=1)
        return {
            "indices": list(topk.indices.detach().cpu().numpy()[0]),
            "probs": list(topk.values.detach().cpu().numpy()[0]),
        }

    # amostragem multinomial
    if tokens_data.shape[0] == 1:
        return torch.multinomial(probabilities, 1).item()
    else:
        return [torch.multinomial(p, 1).item() for p in probabilities]
    
def compare_probabilities(model, token_vector, numerical_vector, dict_descriptive_variables, variables, statistics = ["5%", "95%"], playoff_match = 0, predict_function = predict_num_ngram_model, k = 10):
    
    teams_statistics = [dict_descriptive_variables[key]['mean'] for key in dict_descriptive_variables.keys() if 'home' in key or 'away' in key or 'outcome' in key]
    df_pred_average = pd.DataFrame(predict_function(model, torch.tensor([token_vector]), torch.tensor([numerical_vector + teams_statistics + [playoff_match]]), k = k))\
    .assign(probs = lambda x: 100 * x['probs'])\
    [['indices', 'probs']]\
    .rename(columns = {'probs': 'average'})

    ###############################
    teams_statistics = [dict_descriptive_variables[key][statistics[1]] if key in variables else dict_descriptive_variables[key]['mean'] \
                        for key in dict_descriptive_variables.keys() if 'home' in key or 'away' in key or 'outcome' in key]

    df_pred_high = pd.DataFrame(predict_function(model, torch.tensor([token_vector]), 
                                                      torch.tensor([numerical_vector + teams_statistics + [playoff_match]]), k = k))\
    .assign(probs = lambda x: 100 * x['probs'])\
    [['indices', 'probs']]\
    .rename(columns = {'probs': 'very_high'})
    ###############################
    teams_statistics = [dict_descriptive_variables[key][statistics[0]] if key in variables else dict_descriptive_variables[key]['mean'] \
                        for key in dict_descriptive_variables.keys() if 'home' in key or 'away' in key or 'outcome' in key]

    df_pred_low = pd.DataFrame(predict_function(model, torch.tensor([token_vector]), 
                                                torch.tensor([numerical_vector + teams_statistics + [playoff_match]]), k = k))\
    .assign(probs = lambda x: 100 * x['probs'])\
    [['indices', 'probs']]\
    .rename(columns = {'probs': 'very_low'})
    
    return df_pred_low\
    .merge(df_pred_average, how = 'outer')\
    .merge(df_pred_high, how = 'outer')\
    .sort_values("average", ascending = False)

def plot_loss_over_iterations(trnloss, devloss, ylim = [2, 3]):
    mm_trnloss = [10] + [ma for i, ma in enumerate(moving_average(trnloss, 1000)) if i % 1000 == 1]
    fig, ax = plt.subplots(figsize = (8, 4))
    plt.plot(mm_trnloss, marker = '*')
    plt.plot(devloss[1:], marker = '*')

    ax.set_ylim(ylim)
    [ax.axhline(i, color = 'silver', zorder = -3) for i in np.arange(ylim[0], ylim[1], 0.01)]
    ax.legend(["Training Loss (moving average 1000)", "dev loss (at each 1000 steps)"])
    plt.show()
    return None

def plot_weight_values(model, layer = 'network.0.weight', emb_layer_size = 32, context_size = 5):
    fig, ax = plt.subplots(1, 1, figsize = (8, 4))
    
    pd.DataFrame(model.state_dict()[layer])\
    .describe(percentiles = [0.05, 0.95]).T\
    [['min', '5%', '95%', 'max']]\
    .plot(ax = ax, marker = 'o', markersize = 2)

    [ax.axvline(i - 0.5, linestyle = '--', linewidth = 0.5, color = 'black') for i in [emb_layer_size * i  for i in range(0, context_size + 1)]]
    ax.axhline(0, linestyle = '--', linewidth = 0.5, color = 'black')
    plt.show()
    
    return None

def get_descriptive_values_weights(model, layer = 'network.0.weight', emb_layer_size = 32, context_size = 5, covariables = [], device = "cpu"):
    df_weights = pd.DataFrame()
    
    if device == "cpu":
        weights_ = torch.Tensor.cpu(model.state_dict()[layer]).detach().numpy()
    else:
        weights_ = model.state_dict()[layer].detach().numpy()

    for i in range(1, context_size + 1):
        index1, index2 = (i - 1) * emb_layer_size, i * emb_layer_size



        df_weights_token = pd.DataFrame(weights_)\
        .iloc[:, index1:index2].stack().describe().reset_index().set_index('index').T\
        .assign(amp = lambda x: x['max'] - x['min'])\
        .round(4)\
        .assign(variables = 'token_' + str(6 - i))
    
        df_weights = pd.concat([df_weights, df_weights_token], axis = 0)

    if len(covariables) > 0:
        df_weights_token = pd.DataFrame(weights_)\
        .iloc[:, index2:].describe().T.assign(amp = lambda x: x['max'] - x['min'])\
        .round(4).assign(variables = covariables)

        df_weights = pd.concat([df_weights, df_weights_token], axis = 0)

    return df_weights\
    .set_index('variables')\
    .sort_values("amp", ascending = False)

def get_num_parameters(model):
    total = 0
    for name, param in model.named_parameters():
        total += param.numel()
    return total

def get_num_ngram_model_loss_value(model, x_tok, x_num, y):
    if model.training:
        model.eval()

    output = model(x_tok, x_num)
    baseline_loss = F.cross_entropy(output, y).item()

    return float(baseline_loss)

In [ ]:
def build_logit_mask_ngram_models(df):
    # se limhteam == 0, entao a mascara tem que anular:
    tokens_avoid_limhteam0 = [ 21,  23,  39,  41, 267, 269, 271, 381, 383, 385, 387, 389, 390, 393, 395, 442, 444, 
                            446, 448, 450, 452, 453, 456, 458, 460, 464, 478, 483, 488, 492, 497, 499, 501, 505]

    # se limvteam == 0, entao a mascara tem que anular:
    tokens_avoid_limvteam0 = [ 20,  22,  38,  40, 266, 268, 270, 382, 384, 386, 388, 391, 392, 394, 396, 443, 445, 
                            447, 449, 451, 454, 455, 457, 459, 461, 465, 479, 484, 489, 493, 498, 500, 502, 506]

    # se limhteam == 1, entao a mascara tem que anular:
    tokens_avoid_limhteam1 = [ 15,  17,  19,  31,  33,  35,  37,  51,  53,  57,  59, 139, 161, 253, 255, 263, 265, 
                            365, 367, 368, 371, 372, 373, 377, 378, 397, 399, 401, 403, 405, 407, 409, 411, 412, 
                            415, 416, 419, 420, 421, 422, 426, 428, 430, 432, 434, 435, 438, 440, 462, 476, 480, 
                            481, 486, 490, 494, 496, 503]

    # se limvteam == 1, entao a mascara tem que anular:
    tokens_avoid_limvteam1 = [ 14,  16,  18,  30,  32,  34,  36,  50,  52,  56,  58, 138, 160, 252, 254, 256, 262, 
                            264, 366, 369, 370, 374, 375, 376, 379, 380, 398, 400, 402, 404, 406, 408, 410, 413, 
                            414, 417, 418, 423, 424, 425, 427, 429, 431, 433, 436, 437, 439, 441, 463, 477, 482, 
                            487, 491, 495, 504]

    # se periodo == 1, entao a mascara tem que anular:
    tokens_avoid_period1 = [ 84,  88,  89,  90,  91,  93,  94,  95,  98,  99, 156, 157, 158, 159]

    # se periodo == 2, entao a mascara tem que anular:
    tokens_avoid_period2 = [ 84,  85,  86,  87,  89,  90,  91,  92,  94,  95,  97,  99, 154, 155, 158, 159]

    # se periodo == 3, entao a mascara tem que anular:
    tokens_avoid_period3 = [ 84,  85,  86,  87,  88,  90,  91,  92,  93,  95,  97,  98, 154, 155, 156, 157]

    # se periodo == 4, entao a mascara tem que anular:
    tokens_avoid_period4 = [ 84,  85,  86,  87,  88,  89,  91,  92,  93,  94,  97,  98, 99, 154, 155, 156, 157, 158, 159]

    # se periodo == overtime, entao a mascara tem que anular:
    tokens_avoid_period_ot = [ 85,  86,  87,  88,  89,  90,  92,  93,  94,  95,  97,  98,  99, 154, 155, 156, 157, 158, 159]

    limhfouls = df["limhfouls"].to_numpy()
    limvfouls = df["limvfouls"].to_numpy()
    period = df["period"].to_numpy()

    mask_full = np.zeros((df.shape[0], 601), dtype = bool)

    rows = np.where(limhfouls == 0)[0]
    mask_full[rows[:, None], tokens_avoid_limhteam0] = True

    rows = np.where(limvfouls == 0)[0]
    mask_full[rows[:, None], tokens_avoid_limvteam0] = True

    rows = np.where(limhfouls == 1)[0]
    mask_full[rows[:, None], tokens_avoid_limhteam1] = True

    rows = np.where(limvfouls == 1)[0]
    mask_full[rows[:, None], tokens_avoid_limvteam1] = True

    rows = np.where(period == 1)[0]
    mask_full[rows[:, None], tokens_avoid_period1] = True

    rows = np.where(period == 2)[0]
    mask_full[rows[:, None], tokens_avoid_period2] = True

    rows = np.where(period == 3)[0]
    mask_full[rows[:, None], tokens_avoid_period3] = True

    rows = np.where(period == 4)[0]
    mask_full[rows[:, None], tokens_avoid_period4] = True

    rows = np.where(period > 4)[0]
    mask_full[rows[:, None], tokens_avoid_period_ot] = True

    return mask_full

def build_logit_mask_spent_time_models(df):
    remaining_time = df["remaining_time"].to_numpy()
    mask_full = np.arange(301)[None, :] > remaining_time[:, None]
    return mask_full

